# Stocks Research Agent

This notebook defines a set of tools (skills) and tries it for a sample agent

# 0) IMPORTS

In [1]:
# Setup autoreload - automatically reload modules when they change
%load_ext autoreload
%autoreload 2

In [2]:
# Verify environment setup
import os
import sys

print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")
print(f"\nEnvironment variables loaded:")
print(f"OPENAI_API_KEY: {'✓ Set' if os.getenv('OPENAI_API_KEY') else '✗ Not set'}")
print(f"MASSIVE.COM - ex.POLYGON_API_KEY: {'✓ Set' if os.getenv('POLYGON_API_KEY') else '✗ Not set'}")
print(f"XAI_API_KEY: {'✓ Set' if os.getenv('XAI_API_KEY') else '✗ Not set'}")
print(f"SEC_IDENTITY_EMAIL: {'✓ Set' if os.getenv('SEC_IDENTITY_EMAIL') else '✗ Not set'}")

Python version: 3.12.7 (main, Oct  1 2024, 02:05:46) [Clang 16.0.0 (clang-1600.0.26.3)]
Python executable: /Users/realmistic/Documents/stocks-scoring-agent/.venv/bin/python

Environment variables loaded:
OPENAI_API_KEY: ✓ Set
MASSIVE.COM - ex.POLYGON_API_KEY: ✓ Set
XAI_API_KEY: ✓ Set
SEC_IDENTITY_EMAIL: ✓ Set


In [3]:
# Import key libraries
import yfinance as yf
import pandas as pd
from openai import OpenAI

from pprint import pprint

print("Libraries imported successfully!")

Libraries imported successfully!


# 1) AGENT TOOLS

## 1.0 Ticker (Stock) Info - > many useful stats, incl. some of the important ratios

In [4]:
from typing import Any

def get_company_info(ticker: str) -> dict[str, Any]:
    """
    Get comprehensive company information and fundamental data for a stock ticker.
    
    Returns key metrics organized by category:
    - Company basics: website, industry, sector, employees, officers
    - Price data: current, previous close, day range, 52-week range
    - Market metrics: market cap, volume, beta, PE ratios
    - Valuation: margins, book value, price ratios
    - Ownership: insider/institutional holdings, short interest
    - Analyst data: EPS estimates, targets, recommendations
    - Financial health: cash, returns, growth rates
    
    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')
    
    Returns:
        Dictionary with ticker and organized company information
    """
    try:
        ticker_obj = yf.Ticker(ticker)
        info = ticker_obj.get_info()

        if not info:
            return {"ticker": ticker, "error": "No company info available"}

        # Define the key fields to extract (organized by category)
        key_fields = {
            # Company basics
            "company": ["website", "industry", "sector", "longBusinessSummary",
                        "fullTimeEmployees", "companyOfficers", "region", "fullExchangeName"],

            # Price data
            "price": ["currentPrice", "previousClose", "open", "dayLow", "dayHigh",
                    "regularMarketDayRange", "fiftyTwoWeekLow", "fiftyTwoWeekHigh",
                    "fiftyTwoWeekRange", "allTimeHigh", "allTimeLow"],

            # Market metrics
            "market": ["marketCap", "volume", "averageVolume", "averageVolume10days",
                    "beta", "trailingPE", "forwardPE", "trailingPegRatio"],

            # Moving averages
            "averages": ["fiftyDayAverage", "twoHundredDayAverage",
                        "fiftyDayAverageChange", "twoHundredDayAverageChange"],

            # Valuation ratios
            "valuation": ["priceToSalesTrailing12Months", "priceToBook", "bookValue",
                        "profitMargins", "grossMargins", "ebitdaMargins", "operatingMargins"],

            # Ownership & short interest
            "ownership": ["sharesOutstanding", "floatShares", "sharesPercentSharesOut",
                        "heldPercentInsiders", "heldPercentInstitutions",
                        "sharesShort", "shortRatio", "shortPercentOfFloat"],

            # EPS & earnings
            "earnings": ["trailingEps", "forwardEps", "earningsQuarterlyGrowth",
                        "earningsGrowth", "revenueGrowth", "epsTrailingTwelveMonths",
                        "epsForward", "epsCurrentYear"],

            # Analyst targets & recommendations
            "analyst": ["targetHighPrice", "targetLowPrice", "targetMeanPrice",
                        "targetMedianPrice", "recommendationMean", "recommendationKey",
                        "numberOfAnalystOpinions", "averageAnalystRating"],

            # Financial health
            "financial": ["totalCash", "totalCashPerShare", "totalDebt", "totalRevenue",
                        "freeCashflow", "operatingCashflow", "returnOnAssets",
                        "returnOnEquity", "debtToEquity", "currentRatio", "quickRatio"]
        }

        # Extract data by category
        result = {"ticker": ticker}

        for category, fields in key_fields.items():
            category_data = {}
            for field in fields:
                if field in info:
                    category_data[field] = info[field]
            if category_data:
                result[category] = category_data

        return result

    except Exception as e:
        return {"ticker": ticker, "error": f"Failed to get company info: {str(e)}"}

In [5]:
# Get all company info
info = get_company_info("TSLA")
pprint(info)

{'analyst': {'averageAnalystRating': '2.3 - Buy',
             'numberOfAnalystOpinions': 41,
             'recommendationKey': 'buy',
             'recommendationMean': 2.34043,
             'targetHighPrice': 600.0,
             'targetLowPrice': 123.0,
             'targetMeanPrice': 420.54633,
             'targetMedianPrice': 460.0},
 'averages': {'fiftyDayAverage': 398.3,
              'fiftyDayAverageChange': 8.130005,
              'twoHundredDayAverage': 415.6944,
              'twoHundredDayAverageChange': -9.264404},
 'company': {'companyOfficers': [{'age': 54,
                                  'exercisedValue': 0,
                                  'fiscalYear': 2025,
                                  'maxAge': 1,
                                  'name': 'Mr. Elon R. Musk',
                                  'title': 'Co-Founder, Technoking of Tesla, '
                                           'CEO & Director',
                                  'unexercisedValue': 129602732

In [6]:
# Access specific categories
if "error" not in info:
    # Company basics
    print("\n=== Company Info ===")
    print(f"Industry: {info['company']['industry']}")
    print(f"Sector: {info['company']['sector']}")
    print(f"Employees: {info['company']['fullTimeEmployees']:,}")
    print(f"Website: {info['company']['website']}")

    # Current price & targets
    print("\n=== Price & Targets ===")
    print(f"Current: ${info['price']['currentPrice']:.2f}")
    print(f"52-Week Range: {info['price']['fiftyTwoWeekRange']}")
    print(f"Analyst Target (Mean): ${info['analyst']['targetMeanPrice']:.2f}")
    print(f"Recommendation: {info['analyst']['recommendationKey']}")

    # Key metrics
    print("\n=== Key Metrics ===")
    print(f"Market Cap: ${info['market']['marketCap']:,.0f}")
    print(f"Forward P/E: {info['market']['forwardPE']:.2f}")
    print(f"Profit Margin: {info['valuation']['profitMargins']:.2%}")

    # Ownership
    print("\n=== Ownership ===")
    print(f"Insider: {info['ownership']['heldPercentInsiders']:.2%}")
    print(f"Institutional: {info['ownership']['heldPercentInstitutions']:.2%}")
    print(f"Short Interest: {info['ownership']['shortRatio']}")

    # Growth
    print("\n=== Growth ===")
    print(f"Revenue Growth: {info['earnings']['revenueGrowth']:.2%}")
    print(f"Earnings Growth: {info['earnings']['earningsGrowth']:.2%}")

# Convert a category to DataFrame
if "analyst" in info:
    df = pd.DataFrame([info["analyst"]])
    print(df)



=== Company Info ===
Industry: Auto Manufacturers
Sector: Consumer Cyclical
Employees: 134,785
Website: https://www.tesla.com

=== Price & Targets ===
Current: $406.43
52-Week Range: 288.77 - 498.83
Analyst Target (Mean): $420.55
Recommendation: buy

=== Key Metrics ===
Market Cap: $1,526,438,821,888
Forward P/E: 162.58
Profit Margin: 3.95%

=== Ownership ===
Insider: 11.11%
Institutional: 44.93%
Short Interest: 1.48

=== Growth ===
Revenue Growth: 15.80%
Earnings Growth: 8.30%
   targetHighPrice  targetLowPrice  targetMeanPrice  targetMedianPrice  \
0            600.0           123.0        420.54633              460.0   

   recommendationMean recommendationKey  numberOfAnalystOpinions  \
0             2.34043               buy                       41   

  averageAnalystRating  
0            2.3 - Buy  


## 1.1 EPS trend

In [7]:
from typing import Any

def get_eps_trend(ticker: str) -> dict[str, Any]:
    """
    Get the EPS (Earnings Per Share) trend for a given stock ticker - showing how analyst consensus has changed over time for different periods (quarterly, yearly)
    and diffent points in the past (current, 7daysAgo, 30daysAgo, etc.).
    Index: 0q (This Quarter),  +1q (Next Quarter),  0y (This Year),  +1y (Next Year) 
    and columns showing estimates from different points in the past (current, 7daysAgo, 30daysAgo, etc.). 

    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')
    
    Returns:
        Dictionary with ticker and EPS trend data
    """

    try:
        ticker_obj = yf.Ticker(ticker)
        result = ticker_obj.get_eps_trend()

        if isinstance(result, pd.DataFrame):
            if result.empty:
                return {"ticker": ticker, "error": "No EPS trend data available"}
            result['period']=result.index # create a new column 'period' from the index
            return {"ticker": ticker, "data": result.to_dict(orient='records')}

        # Fallback: wrap unexpected types
        if isinstance(result, dict):
            return {"ticker": ticker, "data": [result]}

        raise TypeError(f"Unexpected return type from get_eps_trend: {type(result)}")

    except Exception as e:
        return {"ticker": ticker, "error": f"Failed to get EPS trend: {str(e)}"}

In [8]:
r = get_eps_trend("TSLA")
# raw output to be used by an agent
print(r)
# pretty output for human consumption
print(f"\nFormatted output for ticker {r['ticker']}:")
print(pd.DataFrame(r['data']))

{'ticker': 'TSLA', 'data': [{'current': 0.45414, '7daysAgo': 0.45396, '30daysAgo': 0.45396, '60daysAgo': 0.44637, '90daysAgo': 0.46212, 'period': '0q'}, {'current': 0.5375, '7daysAgo': 0.5393, '30daysAgo': 0.5393, '60daysAgo': 0.53175, '90daysAgo': 0.5405, 'period': '+1q'}, {'current': 2.05937, '7daysAgo': 2.04967, '30daysAgo': 2.04835, '60daysAgo': 2.03873, '90daysAgo': 2.08858, 'period': '0y'}, {'current': 2.49995, '7daysAgo': 2.50462, '30daysAgo': 2.50969, '60daysAgo': 2.77175, '90daysAgo': 2.81043, 'period': '+1y'}]}

Formatted output for ticker TSLA:
   current  7daysAgo  30daysAgo  60daysAgo  90daysAgo period
0  0.45414   0.45396    0.45396    0.44637    0.46212     0q
1  0.53750   0.53930    0.53930    0.53175    0.54050    +1q
2  2.05937   2.04967    2.04835    2.03873    2.08858     0y
3  2.49995   2.50462    2.50969    2.77175    2.81043    +1y


## 1.2 Earnings dates, previous EPS actual vs. predicted and surprise; next expected earnings date

In [9]:
from typing import Any

def get_earnings_dates(ticker: str) -> dict[str, Any]:
    """
    Get earnings call dates for a stock ticker.
    
    Returns historical earnings data including:
    - Expected EPS
    - Actual EPS  
    - Surprise percentage
    - Earnings dates from multiple quarters and years
    - Next earnings call date
    
    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')

    Returns:
        Dictionary with ticker and earnings dates data, surprise (%) - how reported earnings compared to expectations
    """
    try:
        ticker_obj = yf.Ticker(ticker)
        result = ticker_obj.get_earnings_dates()

        if isinstance(result, pd.DataFrame):
            if result.empty:
                return {"ticker": ticker, "error": "No earnings data available"}

            # Normalize common datetime-like columns to yyyy-mm-dd strings
            result.index = pd.to_datetime(result.index, errors="coerce").strftime("%Y-%m-%d")
            
            # Include the index (dates) as a column before converting to dict
            result = result.reset_index().rename(columns={"index": "date"})
            return {"ticker": ticker, "data": result.to_dict(orient='records')}

        # Unexpected type fallback
        raise TypeError(f"Unexpected return type from get_earnings_dates: {type(result)}")

    except Exception as e:
        return {"ticker": ticker, "error": f"Failed to get earnings dates: {str(e)}"}

In [10]:
r = get_earnings_dates("TSLA")
# raw output to be used by an agent
print(r)
# pretty output for human consumption
print(f"\nFormatted output for ticker {r['ticker']}:")
print(pd.DataFrame(r['data']))

{'ticker': 'TSLA', 'data': [{'Earnings Date': '2026-07-22', 'EPS Estimate': 0.45, 'Reported EPS': nan, 'Surprise(%)': nan}, {'Earnings Date': '2026-04-22', 'EPS Estimate': 0.2, 'Reported EPS': 0.13, 'Surprise(%)': -36.54}, {'Earnings Date': '2026-01-28', 'EPS Estimate': 0.45, 'Reported EPS': 0.5, 'Surprise(%)': 10.96}, {'Earnings Date': '2025-10-22', 'EPS Estimate': 0.56, 'Reported EPS': 0.5, 'Surprise(%)': -10.53}, {'Earnings Date': '2025-07-23', 'EPS Estimate': 0.3, 'Reported EPS': 0.33, 'Surprise(%)': 10.49}, {'Earnings Date': '2025-04-22', 'EPS Estimate': 0.41, 'Reported EPS': 0.27, 'Surprise(%)': -34.89}, {'Earnings Date': '2025-01-29', 'EPS Estimate': 0.77, 'Reported EPS': 0.73, 'Surprise(%)': -4.83}, {'Earnings Date': '2024-10-23', 'EPS Estimate': 0.6, 'Reported EPS': 0.72, 'Surprise(%)': 20.49}, {'Earnings Date': '2024-07-23', 'EPS Estimate': 0.62, 'Reported EPS': 0.52, 'Surprise(%)': -16.15}, {'Earnings Date': '2024-04-23', 'EPS Estimate': 0.49, 'Reported EPS': 0.45, 'Surprise

## 1.3 More of EPS/Growth stats and Analysts estimates

In [11]:
from typing import Any

def get_earnings_analysis(ticker: str) -> dict[str, Any]:
    """
    Get comprehensive earnings and EPS analysis data for a stock ticker.
    
    Combines multiple analyst data sources:
    1. Earnings Estimates - Consensus EPS estimates (avg, low, high, year-ago, analyst count)
    2. EPS Revisions - How analysts have revised estimates (up/down last 7/30 days)
    3. Growth Estimates - Expected earnings growth vs index benchmark
    4. Earnings History - Historical actual vs estimated EPS with surprise %
    
    Period notation:
    - 0q: Current fiscal quarter
    - +1q: Next fiscal quarter
    - 0y: Current fiscal year
    - +1y: Next fiscal year
    - LTG: Long-term growth (growth estimates only)
    
    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')
    
    Returns:
        Dictionary with ticker and all earnings analysis data
    """
    try:
        ticker_obj = yf.Ticker(ticker)
        result = {
            "ticker": ticker,
            "earnings_estimates": None,
            "eps_revisions": None,
            "growth_estimates": None,
            "earnings_history": None
        }

        # 1. Get earnings estimates
        try:
            earnings_est = ticker_obj.get_earnings_estimate()
            if isinstance(earnings_est, pd.DataFrame) and not earnings_est.empty:
                earnings_est = earnings_est.reset_index().rename(columns={"index": "period"})
                result["earnings_estimates"] = earnings_est.to_dict(orient='records')
        except Exception as e:
            result["earnings_estimates"] = {"error": str(e)}

        # 2. Get EPS revisions
        try:
            eps_rev = ticker_obj.get_eps_revisions()
            if isinstance(eps_rev, pd.DataFrame) and not eps_rev.empty:
                eps_rev = eps_rev.reset_index().rename(columns={"index": "period"})
                result["eps_revisions"] = eps_rev.to_dict(orient='records')
        except Exception as e:
            result["eps_revisions"] = {"error": str(e)}

        # 3. Get growth estimates
        try:
            growth_est = ticker_obj.get_growth_estimates()
            if isinstance(growth_est, pd.DataFrame) and not growth_est.empty:
                growth_est = growth_est.reset_index().rename(columns={"index": "period"})
                result["growth_estimates"] = growth_est.to_dict(orient='records')
        except Exception as e:
            result["growth_estimates"] = {"error": str(e)}

        # 4. Get earnings history
        try:
            earnings_hist = ticker_obj.get_earnings_history()
            if isinstance(earnings_hist, pd.DataFrame) and not earnings_hist.empty:
                earnings_hist = earnings_hist.reset_index().rename(columns={"index": "quarter"})
                result["earnings_history"] = earnings_hist.to_dict(orient='records')
        except Exception as e:
            result["earnings_history"] = {"error": str(e)}

        # Check if we got any data at all
        has_data = any(
            result[key] is not None and not isinstance(result[key], dict) or (isinstance(result[key], dict) and "error" not in result[key])
            for key in ["earnings_estimates", "eps_revisions", "growth_estimates", "earnings_history"]
        )

        if not has_data:
            return {"ticker": ticker, "error": "No earnings analysis data available"}

        return result

    except Exception as e:
        return {"ticker": ticker, "error": f"Failed to get earnings analysis: {str(e)}"}


In [12]:

# Get comprehensive earnings analysis
analysis = get_earnings_analysis("TSLA")
pprint(analysis)


{'earnings_estimates': [{'avg': 0.45414,
                         'growth': 0.1353,
                         'high': 0.59,
                         'low': 0.26483,
                         'numberOfAnalysts': 25,
                         'period': '0q',
                         'yearAgoEps': 0.4},
                        {'avg': 0.5375,
                         'growth': 0.075,
                         'high': 0.83,
                         'low': 0.29981,
                         'numberOfAnalysts': 25,
                         'period': '+1q',
                         'yearAgoEps': 0.5},
                        {'avg': 2.05937,
                         'growth': 0.24059999,
                         'high': 3.19,
                         'low': 1.35491,
                         'numberOfAnalysts': 34,
                         'period': '0y',
                         'yearAgoEps': 1.66},
                        {'avg': 2.49995,
                         'growth': 0.2139,
               

## 1.4 Daily historical prices (Open, High, Low, Close, Volume)

In [13]:
from typing import Any

def get_historical_prices(ticker: str, period: str = "2y", interval: str = "1d") -> dict[str, Any]:
    """
    Get historical price data for a stock ticker.
    
    Returns OHLCV (Open, High, Low, Close, Volume) data for the specified period.
    
    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')
        period: Time period to fetch (default: "2y")
                Valid periods: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
        interval: Data interval (default: "1d")
                Valid intervals: 1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d, 5d, 1wk, 1mo, 3mo
    
    Returns:
        Dictionary with ticker and historical price data
    """
    try:
        ticker_obj = yf.Ticker(ticker)
        result = ticker_obj.history(period=period, interval=interval)

        if isinstance(result, pd.DataFrame):
            if result.empty:
                return {"ticker": ticker, "error": f"No historical data available for {ticker}"}

            # Convert datetime index to date strings for cleaner output
            result = result.copy()
            result.index = result.index.date
            result = result.reset_index().rename(columns={"index": "Date"})

            return {
                "ticker": ticker,
                "period": period,
                "interval": interval,
                "data": result.to_dict(orient='records')
            }

        raise TypeError(f"Unexpected return type from history: {type(result)}")

    except Exception as e:
        return {"ticker": ticker, "error": f"Failed to get historical prices: {str(e)}"}


In [14]:
from pprint import pprint
# Get a few days of daily data (NOT a default setting for 2 years)
prices = get_historical_prices("TSLA", period="5d", interval="1d")
pprint(prices)

{'data': [{'Close': 408.95001220703125,
           'Date': datetime.date(2026, 6, 8),
           'Dividends': 0.0,
           'High': 412.94000244140625,
           'Low': 394.7200012207031,
           'Open': 396.3299865722656,
           'Stock Splits': 0.0,
           'Volume': 50328800},
          {'Close': 396.67999267578125,
           'Date': datetime.date(2026, 6, 9),
           'Dividends': 0.0,
           'High': 418.5,
           'Low': 384.239990234375,
           'Open': 411.0299987792969,
           'Stock Splits': 0.0,
           'Volume': 59940200},
          {'Close': 381.5899963378906,
           'Date': datetime.date(2026, 6, 10),
           'Dividends': 0.0,
           'High': 397.0899963378906,
           'Low': 380.1499938964844,
           'Open': 391.5400085449219,
           'Stock Splits': 0.0,
           'Volume': 49695600},
          {'Close': 399.1499938964844,
           'Date': datetime.date(2026, 6, 11),
           'Dividends': 0.0,
           'High': 39

## 1.5 Ticker news (Yahoo Finance)

In [15]:
from typing import Any

def get_ticker_news(ticker: str, count: int = 10) -> dict[str, Any]:
    """
    Get recent news articles for a stock ticker from Yahoo Finance.
    
    Returns key information including title, description, summary, publication date,
    and canonical URL for each news article.
    
    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')
        count: Maximum number of news articles to return (default: 10)
    
    Returns:
        Dictionary with ticker and news data
    """
    try:
        ticker_obj = yf.Ticker(ticker)
        news_raw = ticker_obj.get_news()

        if not news_raw:
            return {"ticker": ticker, "error": "No news available"}

        # Convert to DataFrame to extract content
        news_df = pd.DataFrame(news_raw)

        if 'content' not in news_df.columns:
            return {"ticker": ticker, "error": "Unexpected news data format"}

        # Extract content and convert to DataFrame
        content_df = pd.DataFrame(news_df['content'].tolist())

        # Select only the important fields
        important_fields = ['title', 'description', 'summary', 'pubDate', 'canonicalUrl']

        # Filter to only existing fields
        available_fields = [f for f in important_fields if f in content_df.columns]
        news_filtered = content_df[available_fields]

        # Limit to requested count
        news_filtered = news_filtered.head(count)

        return {
            "ticker": ticker,
            "count": len(news_filtered),
            "data": news_filtered.to_dict(orient='records')
        }

    except Exception as e:
        return {"ticker": ticker, "error": f"Failed to get news: {str(e)}"}


In [16]:

# Get 10 most recent news articles (default)
news = get_ticker_news("TSLA", count=2)
pprint(news)

{'count': 2,
 'data': [{'canonicalUrl': {'lang': 'en-US',
                            'region': 'US',
                            'site': 'finance',
                            'url': 'https://finance.yahoo.com/markets/stocks/articles/tesla-merger-talk-spacex-reshapes-200713727.html'},
           'description': '',
           'pubDate': '2026-06-14T20:07:13Z',
           'summary': 'Speculation around a potential Tesla and SpaceX merger '
                      "has intensified following SpaceX's recent IPO and "
                      'public comments from its president treating the idea as '
                      'a real possibility. Prediction markets, company filings '
                      'and several high profile analysts now treat a merger as '
                      'a credible scenario, rather than a fringe rumor. Any '
                      'move to combine Tesla and SpaceX could reshape how '
                      'investors think about NasdaqGS:TSLA, including its '
         

## 1.6 Ticker news macro (Massive.com (ex. Polygon.io) - 5000 recent news + minsearch RAG)

In [17]:
from typing import Any
import requests
import pandas as pd
import os
from datetime import datetime, timezone
from minsearch import Index
from tqdm import tqdm

# Global variable to store the news index (built once, reused)
_news_index = None
_news_documents = None

def build_polygon_news_index(api_calls: int = 5, news_per_call: int = 1000) -> dict[str, Any]:
    """
    Fetch news from Polygon.io (Massive.com) and build a searchable index.
    
    This should be called once to fetch and index news. The index is stored
    globally and reused by search functions.
    
    Args:
        api_calls: Number of API calls to make (default: 5, fetches ~5000 articles)
        news_per_call: Number of news articles per API call (max: 1000)
    
    Returns:
        Dictionary with status and article count
    """
    global _news_index, _news_documents

    try:
        api_key = os.getenv('POLYGON_API_KEY')
        if not api_key:
            return {"error": "POLYGON_API_KEY not found in environment"}

        now = datetime.now(timezone.utc).strftime("%Y-%m-%d")
        all_news = None
        max_date = now

        print(f"Fetching {api_calls * news_per_call} news articles...")

        for i in tqdm(range(api_calls), desc="API calls"):
            url = f"https://api.massive.com/v2/reference/news?order=desc&limit={news_per_call}&sort=published_utc&published_utc.lt={max_date}&apiKey={api_key}"

            try:
                r = requests.get(url, timeout=10)
                r.raise_for_status()
                data = r.json()

                if 'results' not in data:
                    print(f"No 'results' in response. Keys: {data.keys()}")
                    continue

                cur = pd.json_normalize(data['results'])

                if all_news is None:
                    all_news = cur
                else:
                    all_news = pd.concat([all_news, cur], ignore_index=True)

                max_date = cur.published_utc.min()

            except requests.exceptions.RequestException as e:
                print(f"API call {i+1} failed: {e}")
                continue

        if all_news is None or all_news.empty:
            return {"error": "Failed to fetch news articles"}

        # Convert to documents
        _news_documents = all_news.to_dict(orient='records')

        # Preprocess documents
        print("Preprocessing documents...")
        for doc in tqdm(_news_documents, desc="Converting fields"):
            if isinstance(doc.get('tickers'), list):
                doc['tickers'] = ', '.join(doc['tickers'])
            if isinstance(doc.get('keywords'), list):
                doc['keywords'] = ', '.join(doc['keywords'])

            for field in ['title', 'description', 'author', 'article_url']:
                if doc.get(field) is None:
                    doc[field] = ''
                elif not isinstance(doc.get(field), str):
                    doc[field] = str(doc[field])

        # Build index
        print("Building search index...")
        _news_index = Index(
            text_fields=["title", "description", "keywords", "author", "tickers", "article_url"],
            keyword_fields=["published_utc", "publisher.name"]
        )
        _news_index.fit(_news_documents)

        return {
            "status": "success",
            "articles_indexed": len(_news_documents),
            "message": f"Index built with {len(_news_documents)} articles"
        }

    except Exception as e:
        return {"error": f"Failed to build news index: {str(e)}"}


In [18]:
def search_news_by_ticker(ticker: str, num_results: int = 30) -> dict[str, Any]:
    """
    Search indexed news articles for a specific stock ticker.
    
    Searches across title, description, keywords, and tickers with boosting
    that prioritizes ticker matches.
    
    Args:
        ticker: Stock ticker symbol (e.g., 'TSLA', 'AAPL', 'GOOGL')
        num_results: Maximum number of results to return (default: 30)
    
    Returns:
        Dictionary with ticker and matching news articles
    """
    global _news_index

    if _news_index is None:
        return {
            "ticker": ticker,
            "error": "News index not built. Call build_polygon_news_index() first."
        }

    try:
        results = _news_index.search(
            query=ticker,
            num_results=num_results,
            boost_dict={
                'tickers': 5.0,      # Highest boost for ticker field
                'title': 3.0,        # High boost for title
                'description': 1.5,  # Medium boost for description
                'keywords': 1.0      # Standard boost for keywords
            }
        )

        return {
            "ticker": ticker,
            "count": len(results),
            "data": results
        }

    except Exception as e:
        return {"ticker": ticker, "error": f"Search failed: {str(e)}"}


def search_news_by_query(query: str, num_results: int = 30) -> dict[str, Any]:
    """
    Search indexed news articles by free-text query.
    
    Searches across title, description, keywords, and tickers with boosting
    that prioritizes description and keyword matches.
    
    Args:
        query: Search query (e.g., 'Tesla competitors EV market', 'AI robotics')
        num_results: Maximum number of results to return (default: 30)
    
    Returns:
        Dictionary with query and matching news articles
    """
    global _news_index

    if _news_index is None:
        return {
            "query": query,
            "error": "News index not built. Call build_polygon_news_index() first."
        }

    try:
        results = _news_index.search(
            query=query,
            num_results=num_results,
            boost_dict={
                'tickers': 1.0,       # Standard boost for ticker field
                'title': 3.0,         # High boost for title
                'description': 5.0,   # Highest boost for description
                'keywords': 5.0       # Highest boost for keywords
            }
        )

        return {
            "query": query,
            "count": len(results),
            "data": results
        }

    except Exception as e:
        return {"query": query, "error": f"Search failed: {str(e)}"}


In [19]:
# USAGE EXAMPLE
# Step 1: Build the index once (takes a few minutes)
result = build_polygon_news_index(api_calls=5)
pprint(result)

Fetching 5000 news articles...


API calls: 100%|██████████| 5/5 [00:05<00:00,  1.19s/it]


Preprocessing documents...


Converting fields: 100%|██████████| 5000/5000 [00:00<00:00, 1073920.52it/s]


Building search index...
{'articles_indexed': 5000,
 'message': 'Index built with 5000 articles',
 'status': 'success'}


In [20]:
# Step 2: Search by ticker
tsla_news = search_news_by_ticker("TSLA", num_results=10)
print(f"\nFound {tsla_news['count']} articles for {tsla_news['ticker']}")
for i, article in enumerate(tsla_news['data'][:5], 1):
    print(f"{i}. {article['title']}")
    print(f"   Published: {article['published_utc']}")
    print(f"   Tickers: {article['tickers']}")
    print(f"   Description: {article['description']}")
    print(f"   Article URL: {article['article_url']}")


Found 10 articles for TSLA
1. Crypto Markets Are Already Pricing SPCX: What Happens At IPO?
   Published: 2026-06-04T16:09:45Z
   Tickers: TSLA
   Description: SpaceX is preparing for a major IPO in June 2026 targeting a $1.8 trillion valuation and $70-75 billion in fundraising. Crypto markets have already begun pricing the event through synthetic derivatives on platforms like Hyperliquid and Trade.xyz, with SPCX contracts rising 44% since launch. However, regulatory concerns persist regarding information gaps, investor protections, and the classification of these synthetic pre-IPO products.
   Article URL: https://www.benzinga.com/Opinion/26/06/53007825/crypto-markets-are-already-pricing-spcx-what-happens-at-ipo?utm_source=benzinga_taxonomy&utm_medium=rss_feed_free&utm_content=taxonomy_rss&utm_campaign=channel
2. Will SpaceX Merge With Tesla? Here's What Prediction Markets Say.
   Published: 2026-06-08T05:31:00Z
   Tickers: TSLA
   Description: Prediction markets are showing increase

In [21]:
# Step 3: Search by query
query = "Tesla competitors EV margins autonomy AI robotics"
results = search_news_by_query(query, num_results=25)
print(f"\nFound {results['count']} articles for query: '{results['query']}'")
for i, article in enumerate(results['data'][:5], 1):
    print(f"{i}. {article['title']}")
    print(f"   Published: {article['published_utc']}")
    print(f"   Tickers: {article['tickers']}")
    print(f"   Description: {article['description']}")
    print(f"   Article URL: {article['article_url']}")


Found 25 articles for query: 'Tesla competitors EV margins autonomy AI robotics'
1. EV Sales Skyrocket: Tesla Stock Surges On 67% European Boom, Blockbuster SpaceX IPO Rumors
   Published: 2026-05-27T15:40:06Z
   Tickers: TSLA, BYDDY, GELHY
   Description: Tesla stock gained 1.11% on Wednesday, driven by strong European EV sales growth (67.2% year-over-year in April) and speculation about a potential future merger with SpaceX. The SpaceX IPO is expected on Nasdaq under ticker SPCX following its merger with xAI, valued at $1.25 trillion. Tesla continues expanding beyond traditional auto manufacturing into AI, robotics, and energy sectors, with the next earnings catalyst expected on July 22.
   Article URL: https://www.benzinga.com/markets/tech/26/05/52813754/ev-sales-skyrocket-tesla-stock-surges-on-67-european-boom-blockbuster-spacex-ipo-rumors?utm_source=benzinga_taxonomy&utm_medium=rss_feed_free&utm_content=taxonomy_rss&utm_campaign=channel
2. Why Tesla Robotaxi Dreams Can’t Rescue T

## 1.7 Macro stats (market cap) - 10k companies (several csvs to save and query) 

  Key features:
  - ✅ Loads and merges all 4 databases (10k+ companies)
  - ✅ Unified search across market cap, P/E, dividend, and operating margin
  - ✅ Pre-built filters for value and growth investing strategies
  - ✅ Cache all databases for fast repeated queries
  - ✅ Multiple search criteria can be combined
  - ✅ Country filtering


In [22]:
from typing import Any, Optional
import requests
import pandas as pd
from io import StringIO

# Global variables to cache the databases
_companies_marketcap_db = None
_companies_pe_db = None
_companies_dividend_db = None
_companies_margin_db = None
_unified_db = None

def load_all_companies_databases(force_refresh: bool = False) -> dict[str, Any]:
    """
    Download and load all company databases from companiesmarketcap.com.
    
    Loads 4 databases:
    1. Market Cap - Top companies by market capitalization
    2. P/E Ratio - Top companies by price-to-earnings ratio
    3. Dividend Yield - Top companies by dividend yield percentage
    4. Operating Margin - Top companies by operating margin percentage
    
    The databases are cached globally and merged by ticker symbol for unified searching.
    Use force_refresh=True to re-download.
    
    Args:
        force_refresh: If True, re-download all databases even if cached
    
    Returns:
        Dictionary with status and database info
    """
    global _companies_marketcap_db, _companies_pe_db, _companies_dividend_db
    global _companies_margin_db, _unified_db

    # Return cached data if available
    if _unified_db is not None and not force_refresh:
        df = pd.DataFrame(_unified_db)
        return {
            "status": "loaded_from_cache",
            "total_companies": len(_unified_db),
            "available_columns": list(df.columns),
            "message": f"All databases loaded from cache"
        }

    try:
        databases = {
            "marketcap": "https://companiesmarketcap.com/usd/?download=csv",
            "pe_ratio": "https://companiesmarketcap.com/top-companies-by-pe-ratio/?download=csv",
            "dividend": "https://companiesmarketcap.com/top-companies-by-dividend-yield/?download=csv",
            "margin": "https://companiesmarketcap.com/top-companies-by-operating-margin/?download=csv"
        }

        loaded = {}

        for name, url in databases.items():
            print(f"Downloading {name} database...")
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                csv_data = StringIO(response.text)
                df = pd.read_csv(csv_data)
                loaded[name] = df
                print(f"✓ Loaded {len(df)} companies from {name}")
            except Exception as e:
                print(f"✗ Failed to load {name}: {e}")
                loaded[name] = None

        # Store individual databases
        _companies_marketcap_db = loaded["marketcap"]
        _companies_pe_db = loaded["pe_ratio"]
        _companies_dividend_db = loaded["dividend"]
        _companies_margin_db = loaded["margin"]

        # Merge all databases by Symbol for unified view
        print("\nMerging databases...")

        # Start with market cap as base
        unified = loaded["marketcap"].copy()

        # Merge P/E ratio - only keep pe_ratio_ttm column
        if loaded["pe_ratio"] is not None and 'pe_ratio_ttm' in loaded["pe_ratio"].columns:
            pe_df = loaded["pe_ratio"][['Symbol', 'pe_ratio_ttm']]
            unified = unified.merge(pe_df, on='Symbol', how='left')
            print(f"✓ Added P/E ratio column")

        # Merge Dividend yield - only keep dividend_yield_ttm column
        if loaded["dividend"] is not None and 'dividend_yield_ttm' in loaded["dividend"].columns:
            div_df = loaded["dividend"][['Symbol', 'dividend_yield_ttm']]
            # Convert percentage to decimal
            div_df['dividend_yield_ttm'] = div_df['dividend_yield_ttm'] / 100.0 
            unified = unified.merge(div_df, on='Symbol', how='left')
            print(f"✓ Added Dividend yield column")

        # Merge Operating margin - only keep operating_margin_ttm column
        if loaded["margin"] is not None and 'operating_margin_ttm' in loaded["margin"].columns:
            margin_df = loaded["margin"][['Symbol', 'operating_margin_ttm']]
            # Convert percentage to decimal
            margin_df['operating_margin_ttm'] = margin_df['operating_margin_ttm']/100.0
            unified = unified.merge(margin_df, on='Symbol', how='left')
            print(f"✓ Added Operating margin column")

        print(f"\nFinal columns: {list(unified.columns)}")

        _unified_db = unified.to_dict(orient='records')

        return {
            "status": "success",
            "databases_loaded": {
                "marketcap": len(loaded["marketcap"]) if loaded["marketcap"] is not None else 0,
                "pe_ratio": len(loaded["pe_ratio"]) if loaded["pe_ratio"] is not None else 0,
                "dividend": len(loaded["dividend"]) if loaded["dividend"] is not None else 0,
                "margin": len(loaded["margin"]) if loaded["margin"] is not None else 0
            },
            "total_companies": len(_unified_db),
            "available_columns": list(unified.columns),
            "message": f"All databases merged with {len(_unified_db)} unique companies"
        }

    except Exception as e:
        return {"error": f"Failed to load databases: {str(e)}"}


def get_available_columns() -> dict[str, Any]:
    """
    Get list of available columns in the unified database.
    
    Returns:
        Dictionary with available column names
    """
    global _unified_db

    if _unified_db is None:
        return {"error": "Database not loaded. Call load_all_companies_databases() first."}

    df = pd.DataFrame(_unified_db)
    return {
        "columns": list(df.columns),
        "total_columns": len(df.columns)
    }


def search_companies(
    query: Optional[str] = None,
    ticker: Optional[str] = None,
    min_market_cap: Optional[float] = None,
    max_market_cap: Optional[float] = None,
    min_pe: Optional[float] = None,
    max_pe: Optional[float] = None,
    min_dividend: Optional[float] = None,
    max_dividend: Optional[float] = None,
    min_margin: Optional[float] = None,
    max_margin: Optional[float] = None,
    country: Optional[str] = None,
    limit: int = 50
) -> dict[str, Any]:
    """
    Search companies across all databases with comprehensive filtering.
    
    Args:
        query: Search by company name (case-insensitive partial match)
        ticker: Search by exact ticker symbol
        min_market_cap: Minimum market cap in USD
        max_market_cap: Maximum market cap in USD
        min_pe: Minimum P/E ratio
        max_pe: Maximum P/E ratio
        min_dividend: Minimum dividend yield (%)
        max_dividend: Maximum dividend yield (%)
        min_margin: Minimum operating margin (%)
        max_margin: Maximum operating margin (%)
        country: Filter by country (e.g., 'USA', 'China')
        limit: Maximum number of results (default: 50)
    
    Returns:
        Dictionary with matching companies and all available metrics
    """
    global _unified_db

    if _unified_db is None:
        return {"error": "Database not loaded. Call load_all_companies_databases() first."}

    try:
        df = pd.DataFrame(_unified_db)

        # Apply filters
        if ticker:
            df = df[df['Symbol'].str.upper() == ticker.upper()]

        if query:
            df = df[df['Name'].str.contains(query, case=False, na=False)]

        if min_market_cap is not None:
            df = df[df['marketcap'] >= min_market_cap]

        if max_market_cap is not None:
            df = df[df['marketcap'] <= max_market_cap]

        if 'pe_ratio_ttm' in df.columns:
            if min_pe is not None:
                df = df[pd.to_numeric(df['pe_ratio_ttm'], errors='coerce') >= min_pe]
            if max_pe is not None:
                df = df[pd.to_numeric(df['pe_ratio_ttm'], errors='coerce') <= max_pe]

        if 'dividend_yield_ttm' in df.columns:
            if min_dividend is not None:
                df = df[pd.to_numeric(df['dividend_yield_ttm'], errors='coerce') >= min_dividend]
            if max_dividend is not None:
                df = df[pd.to_numeric(df['dividend_yield_ttm'], errors='coerce') <= max_dividend]

        if 'operating_margin_ttm' in df.columns:
            if min_margin is not None:
                df = df[pd.to_numeric(df['operating_margin_ttm'], errors='coerce') >= min_margin]
            if max_margin is not None:
                df = df[pd.to_numeric(df['operating_margin_ttm'], errors='coerce') <= max_margin]

        if country:
            df = df[df['country'].str.upper() == country.upper()]

        # Limit results
        df = df.head(limit)

        if df.empty:
            return {
                "count": 0,
                "message": "No companies found matching criteria"
            }

        return {
            "count": len(df),
            "data": df.to_dict(orient='records')
        }

    except Exception as e:
        return {"error": f"Search failed: {str(e)}"}


def get_top_value_companies(
    min_dividend: float = 2.0/100.0, # 2%
    max_pe: float = 25,
    min_margin: float = 10/100.0, # 10%
    min_market_cap: float = 1_000_000_000,
    limit: int = 50
) -> dict[str, Any]:
    """
    Find potential value companies based on fundamental criteria.
    
    Default criteria:
    - Dividend yield >= 2%
    - P/E ratio <= 25
    - Operating margin >= 10%
    - Market cap >= $1B
    
    Args:
        min_dividend: Minimum dividend yield %
        max_pe: Maximum P/E ratio
        min_margin: Minimum operating margin %
        min_market_cap: Minimum market cap
        limit: Maximum results
    
    Returns:
        Dictionary with companies meeting value criteria
    """
    return search_companies(
        min_dividend=min_dividend,
        max_pe=max_pe,
        min_margin=min_margin,
        min_market_cap=min_market_cap,
        limit=limit
    )


def get_top_growth_companies(
    min_margin: float = 20/100.0, # 20%
    max_pe: Optional[float] = None,
    min_market_cap: float = 1_000_000_000,
    limit: int = 50
) -> dict[str, Any]:
    """
    Find potential growth companies based on fundamental criteria.
    
    Default criteria:
    - Operating margin >= 20% (high profitability)
    - Market cap >= $1B
    
    Args:
        min_margin: Minimum operating margin %
        max_pe: Maximum P/E ratio (optional)
        min_market_cap: Minimum market cap
        limit: Maximum results
    
    Returns:
        Dictionary with companies meeting growth criteria
    """
    return search_companies(
        min_margin=min_margin,
        max_pe=max_pe,
        min_market_cap=min_market_cap,
        limit=limit
    )



In [23]:
# USAGE EXAMPLE

# Load databases
result = load_all_companies_databases(force_refresh=True)
pprint(result)

# Search for value stocks
value_stocks = get_top_value_companies(min_dividend=3.0/100.0, max_pe=20, min_margin=15/100.0, limit=25)
print(f"\nFound {value_stocks['count']} value stocks")

if "data" in value_stocks:
    df = pd.DataFrame(value_stocks['data'])
    print(df[['Name', 'Symbol', 'marketcap', 'pe_ratio_ttm', 'dividend_yield_ttm', 'operating_margin_ttm']])


✓ Loaded 10879 companies from marketcap
✓ Loaded 10879 companies from pe_ratio
✓ Loaded 10879 companies from dividend
✓ Loaded 10879 companies from margin

Merging databases...
✓ Added P/E ratio column
✓ Added Dividend yield column
✓ Added Operating margin column

Final columns: ['Rank', 'Name', 'Symbol', 'marketcap', 'price (USD)', 'country', 'pe_ratio_ttm', 'dividend_yield_ttm', 'operating_margin_ttm']
{'available_columns': ['Rank',
                       'Name',
                       'Symbol',
                       'marketcap',
                       'price (USD)',
                       'country',
                       'pe_ratio_ttm',
                       'dividend_yield_ttm',
                       'operating_margin_ttm'],
 'databases_loaded': {'dividend': 10879,
                      'margin': 10879,
                      'marketcap': 10879,
                      'pe_ratio': 10879},
 'message': 'All databases merged with 10879 unique companies',
 'status': 'success',
 'total

/var/folders/_6/616g1v7j04jdsf8gy64v_t640000gn/T/ipykernel_30407/3381459230.py:90: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  div_df['dividend_yield_ttm'] = div_df['dividend_yield_ttm'] / 100.0
/var/folders/_6/616g1v7j04jdsf8gy64v_t640000gn/T/ipykernel_30407/3381459230.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  margin_df['operating_margin_ttm'] = margin_df['operating_margin_ttm']/100.0


In [24]:
# RAW DATA in the combined database
# checking the unified database of al
df = pd.DataFrame(_unified_db)

In [25]:
df.head()

,Rank,Name,Symbol,marketcap,price (USD),country,pe_ratio_ttm,dividend_yield_ttm,operating_margin_ttm
0,1,NVIDIA,NVDA,4969906831360,205.19,United States,41.5811,0.019513,65.62
1,2,Alphabet (Google),GOOG,4367737946112,358.16,United States,27.1492,0.233687,46.31
2,3,Apple,AAPL,4275929874432,291.13,United States,34.9825,0.362063,31.89
3,4,Microsoft,MSFT,2902586556416,390.74,United States,22.9649,0.684827,48.54
4,5,Amazon,AMZN,2566108479488,238.55,United States,27.8559,0.000000,13.57


## 1.8) SEC filings (Edgar)

In [26]:
from typing import Any, Literal
import os

from edgar import Company, set_identity


def get_sec_filing(
    ticker: str,
    filing_type: Literal["annual", "quarterly"] = "quarterly",
    periods_ago: int = 0,
) -> dict[str, Any]:
    """
    Get the text of an SEC filing.

    Args:
        ticker: Stock ticker (e.g. AAPL, MSFT, TSLA)
        filing_type:
            - "quarterly" -> 10-Q
            - "annual" -> 10-K
        periods_ago:
            Which filing to retrieve.
            0 = latest
            1 = previous filing
            2 = two filings ago
            ...

    Returns:
        Dictionary containing filing metadata and text.
    """

    try:
        sec_email = os.getenv("SEC_IDENTITY_EMAIL")
        if not sec_email:
            return {
                "success": False,
                "error": "SEC_IDENTITY_EMAIL not set in environment",
            }

        set_identity(sec_email)

        form = {
            "quarterly": "10-Q",
            "annual": "10-K",
        }[filing_type]

        company = Company(ticker)

        filings = company.get_filings(form=form)

        filing_list = list(filings)

        if periods_ago >= len(filing_list):
            return {
                "success": False,
                "ticker": ticker,
                "filing_type": filing_type,
                "periods_ago": periods_ago,
                "available_filings": len(filing_list),
                "error": f"Only {len(filing_list)} {form} filings available.",
            }

        filing = filing_list[periods_ago]
        text = filing.text()

        return {
            "success": True,
            "ticker": ticker,
            "filing_type": filing_type,
            "form": form,
            "periods_ago": periods_ago,
            "filing_date": getattr(filing, "filing_date", None),
            "accession_number": getattr(filing, "accession_number", None),
            "text_length": len(text),
            "text": text,
        }

    except Exception as e:
        return {
            "success": False,
            "ticker": ticker,
            "filing_type": filing_type,
            "periods_ago": periods_ago,
            "error": str(e),
        }

In [27]:
# DEBUG: testing the function to get SEC filings

# # Latest quarterly filing
# get_sec_filing("AAPL")

# # Previous quarterly filing
# get_sec_filing("AAPL", periods_ago=1)

# # Latest annual report
# get_sec_filing("AAPL", filing_type="annual")

# 3 annual reports ago
get_sec_filing("AAPL", filing_type="annual", periods_ago=3)

{'success': True,
 'ticker': 'AAPL',
 'filing_type': 'annual',
 'form': '10-K',
 'periods_ago': 3,
 'filing_date': datetime.date(2022, 10, 28),
 'accession_number': '0000320193-22-000108',
 'text_length': 265739,
 'text': 'UNITED STATES\n\nSECURITIES AND EXCHANGE COMMISSION\n\nWashington, D.C. 20549\n\nFORM 10-K\n\n(Mark One)\n\n☒ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the fiscal year ended September 24, 2022\n\nor\n\n☐TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the transition period from to.\n\nCommission File Number: 001-36743\n\nApple Inc.\n\n(Exact name of Registrant as specified in its charter)\n\n  California                                    94-2404110                            \n  (State or other jurisdiction                  (I.R.S. Employer Identification No.)  \n  One Apple Park Way                                                                  \n  Cupertino, Californi

##  1.9) [PAID via X.AI] X/Twitter posts by engagement (sorted by impact)

In [28]:
from typing import Any
import os
from datetime import datetime, timedelta
import openai

def get_twitter_posts_by_engagement(
    ticker: str, 
    max_posts: int = 5, 
    days_back: int = 1,
    min_engagement: int = 100
) -> dict[str, Any]:
    """
    Get Twitter/X posts about a stock, sorted by HIGHEST ENGAGEMENT (viral posts).
    
    Engagement = views + likes + replies + retweets + bookmarks
    Posts are sorted by total engagement to find the most impactful discussions.
    
    Args:
        ticker: Stock ticker symbol (e.g., 'TSLA', 'AAPL')
        max_posts: Number of posts to return (default: 5)
        days_back: How many days to look back (default: 1)
        min_engagement: Minimum engagement threshold (default: 100)
    
    Returns:
        Dictionary with ticker, posts data, and metadata
    """
    api_key = os.getenv('XAI_API_KEY')
    if not api_key:
        return {
            'success': False,
            'ticker': ticker,
            'error': 'XAI_API_KEY not found in environment'
        }
    
    try:
        client = openai.OpenAI(
            api_key=api_key,
            base_url="https://api.x.ai/v1"
        )
        
        # Calculate date range
        to_date = datetime.now()
        from_date = to_date - timedelta(days=days_back)
        
        search_query = f"""
        Search X/Twitter for HIGH-ENGAGEMENT posts about ${ticker} stock from the last {days_back} day(s).
        
        **CRITICAL SORTING REQUIREMENT:**
        - Sort by HIGHEST ENGAGEMENT first (most viral/impactful)
        - Engagement score = views + likes + replies + retweets + bookmarks
        - Only show posts with {min_engagement}+ total engagement
        
        **For each of the top {max_posts} posts, provide:**
        1. **Engagement Metrics** (prominently displayed):
           - Total engagement score
           - Views, Likes, Replies, Retweets, Bookmarks (breakdown)
        2. **Post Content**: Full text
        3. **Author**: Username and verification status
        4. **Timestamp**: Exact time posted
        5. **Link**: Working X.com URL
        6. **Sentiment**: Bullish/Bearish/Neutral with reasoning
        7. **Why it matters**: Why this post is significant/viral
        
        **Focus on:**
        - Posts from verified accounts, analysts, or influencers
        - Posts that generated real discussion/impact
        - News reactions, earnings discussion, technical analysis
        - Skip spam, bots, or low-quality posts
        
        Date range: {from_date.strftime('%Y-%m-%d')} to {to_date.strftime('%Y-%m-%d')}
        
        Return the {max_posts} MOST VIRAL posts sorted by engagement score (highest first).
        """
        
        response = client.responses.create(
            model="grok-4.3",
            input=search_query,
            tools=[{"type": "x_search"}]
        )
        
        # Extract content
        content = ""
        for item in response.output:
            if item.type == "message":
                for content_block in item.content:
                    if content_block.type == "output_text":
                        content += content_block.text
        
        return {
            'success': True,
            'ticker': ticker,
            'response': content,
            'search_params': {
                'max_posts': max_posts,
                'days_back': days_back,
                'min_engagement': min_engagement,
                'date_range': f"{from_date.strftime('%Y-%m-%d')} to {to_date.strftime('%Y-%m-%d')}",
                'sort_by': 'engagement_desc'
            },
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'platform': 'Twitter/X'
        }
        
    except Exception as e:
        return {
            'success': False,
            'ticker': ticker,
            'error': f'Twitter search failed: {str(e)}'
        }

In [29]:
# testing the function to get Twitter posts by engagement
test_twitter = get_twitter_posts_by_engagement("TSLA", max_posts=3, days_back=5, min_engagement=200)

In [30]:
print(test_twitter['response'])

**Top 3 Most Viral $TSLA Posts (Last 5 Days: 2026-06-09 to 2026-06-14)**  
Sorted by highest engagement score (views + likes + replies + retweets + bookmarks). Only posts with 200+ total engagement included. All from verified/influencer accounts with real discussion.

### 1. **TrendSpider** (@TrendSpider)  
**Engagement Metrics** (Total: **145,943**)  
- Views: 145,512  
- Likes: 357  
- Replies: 23  
- Retweets: 17  
- Bookmarks: 34  

**Post Content**: Elon is one of one $TSLA $SPCX (with chart image showing technical pattern).  

**Author**: @TrendSpider (verified trading platform account, award-winning charting tool).  

**Timestamp**: Fri, 12 Jun 2026 16:45:00 GMT  

**Link**: https://x.com/TrendSpider/status/2065475465071202705  

**Sentiment**: **Bullish** – Highlights Elon Musk’s unique influence on Tesla as a positive catalyst.  

**Why it matters**: Extremely high views indicate massive reach; the post went viral for its bold technical analysis tying Elon directly to $TSLA mo

##  1.10) [PAID via X.AI] Reddit posts by engagement (sorted by impact)

In [31]:
def get_reddit_discussions_by_impact(
    ticker: str,
    max_posts: int = 5,
    days_back: int = 7,
    min_upvotes: int = 50
) -> dict[str, Any]:
    """
    Get Reddit discussions about a stock, sorted by HIGHEST IMPACT.
    
    Impact = upvotes + comments + awards (weighted)
    Searches major investment subreddits for quality discussions.
    
    Args:
        ticker: Stock ticker symbol (e.g., 'TSLA', 'AAPL')
        max_posts: Number of posts to return (default: 5)
        days_back: How many days to look back (default: 7)
        min_upvotes: Minimum upvotes threshold (default: 50)
    
    Returns:
        Dictionary with ticker, posts data, and metadata
    """
    api_key = os.getenv('XAI_API_KEY')
    if not api_key:
        return {
            'success': False,
            'ticker': ticker,
            'error': 'XAI_API_KEY not found in environment'
        }
    
    try:
        client = openai.OpenAI(
            api_key=api_key,
            base_url="https://api.x.ai/v1"
        )
        
        reddit_query = f"""
        Search Reddit for HIGH-IMPACT discussions about ${ticker} stock from the last {days_back} days.
        
        **TARGET SUBREDDITS (search these specifically):**
        - r/investing (serious investment analysis)
        - r/stocks (general stock discussion)
        - r/SecurityAnalysis (fundamental analysis)
        - r/ValueInvesting (value perspective)
        - r/wallstreetbets (retail sentiment & options activity)
        - r/StockMarket (market discussion)
        - Ticker-specific subs if they exist (e.g., r/TeslaInvestorsClub for TSLA)
        
        **CRITICAL SORTING REQUIREMENT:**
        - Sort by HIGHEST IMPACT score first
        - Impact score = upvotes + (comments × 2) + (awards × 5)
        - Only posts with {min_upvotes}+ upvotes
        
        **CONTENT PRIORITIES (in order):**
        1. DD (Due Diligence) posts with financial analysis
        2. Earnings reaction threads with substantial discussion
        3. News reaction posts with quality comments
        4. Technical/fundamental analysis
        5. Catalyst discussions (product launches, regulatory news, etc.)
        
        **For each of the top {max_posts} posts, provide:**
        1. **Subreddit & Title**
        2. **Impact Metrics**:
           - Upvotes
           - Comments count
           - Awards (Gold, Silver, Helpful, etc.)
           - Impact score calculation
        3. **Content Summary**: Post summary + key insights from top comments
        4. **Sentiment**: Bullish/Bearish/Neutral with reasoning
        5. **Key Takeaways**: 2-3 main investment insights
        6. **Quality Level**: High/Medium (based on analysis depth)
        7. **Link**: Direct Reddit URL
        8. **Author credibility**: Note if author has history of quality posts
        
        **Focus on:**
        - Posts that provide real investment insights
        - Quality discussions in comments
        - Fundamental or technical analysis
        - Skip memes, low-effort posts, pure speculation
        
        Return the {max_posts} HIGHEST IMPACT posts sorted by impact score (highest first).
        Filter for posts from the last {days_back} days only.
        """
        
        response = client.responses.create(
            model="grok-4.3",
            input=reddit_query,
            tools=[{"type": "web_search"}]
        )
        
        # Extract content
        content = ""
        for item in response.output:
            if item.type == "message":
                for content_block in item.content:
                    if content_block.type == "output_text":
                        content += content_block.text
        
        return {
            'success': True,
            'ticker': ticker,
            'response': content,
            'search_params': {
                'max_posts': max_posts,
                'days_back': days_back,
                'min_upvotes': min_upvotes,
                'sort_by': 'impact_score_desc'
            },
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'platform': 'Reddit'
        }
        
    except Exception as e:
        return {
            'success': False,
            'ticker': ticker,
            'error': f'Reddit search failed: {str(e)}'
        }

In [32]:
test_reddit = get_reddit_discussions_by_impact("TSLA", max_posts=3, days_back=7, min_upvotes=100) 

In [33]:
print(test_reddit['response'])

**Top post (by far the highest impact in results):**

**1. Subreddit & Title:** r/wallstreetbets – "SpaceX is buying Tesla - Double Down"  
**Impact Metrics:**  
- Upvotes: 1,651  
- Comments: 258  
- Awards: Not specified in available data (assume 0 for calculation)  
- Impact score: 1,651 + (258 × 2) = 2,167 (highest by a wide margin among recent results)  

**Content Summary:** The post details a speculative options bet on a potential SpaceX acquisition of Tesla (Tesla shareholders receiving SpaceX shares). It argues Elon Musk could use this to secure control over AI/robotics (Optimus) amid voting control issues at Tesla, with post-announcement trading lock-step until close (~6 months). It highlights specific deep out-of-the-money calls as highly profitable in a surprise deal scenario. Top comments discuss how this could help Musk earn his Tesla pay package (EBITDA hurdles) and note related SpaceX/Tesla dynamics.[[1]](https://www.reddit.com/r/wallstreetbets/comments/1u0bojo/spacex_i

# 2) Building an agent 

## 2.0) A completely FREE agent based on the Qwen3 model with function calling (via Ollama)
* Idea: one shot call agent with thinking and suggested path of analysis
* https://docs.ollama.com/capabilities/tool-calling#python : "Single-shot vs. Thinking tool calling"
* https://ollama.com/library/qwen3:32b : "Expertise in agent capabilities"
* https://qwen.ai/blog?id=qwen3  "Qwen3 excels in tool calling capabilities"

In [34]:
from ollama import chat, ChatResponse

In [35]:
# # only Yahoo finance
available_functions = {
  'get_company_info': get_company_info,
  'get_eps_trend': get_eps_trend,
  'get_earnings_dates': get_earnings_dates,
  'get_earnings_analysis': get_earnings_analysis,
  'get_historical_prices': get_historical_prices,
  'get_ticker_news': get_ticker_news,
  'search_news_by_ticker': search_news_by_ticker,
  'search_news_by_query': search_news_by_query,
  'get_sec_filing': get_sec_filing
}

In [36]:
# Enhanced system message with tool suggestions
system_message = """You are a financial analysis assistant with access to multiple data sources:

1. **Yahoo Finance Tools**: Company info, EPS trends, earnings dates, historical prices
2. **News Database (5000 recent articles)**: Search by ticker or query to find market sentiment, industry trends, competitive landscape
3. **SEC Filings**: 10-Q (quarterly) and 10-K (annual) reports for detailed company analysis

**Suggested Analysis Workflow:**
- Start with company fundamentals (get_company_info)
- Check recent SEC filings (at least last annual and quarterly) for strategic insights, catalysts, and risk factors
- Search news database (search_news_by_ticker, search_news_by_query) for:
  * Market sentiment and vertical/industry trends
  * Competitive positioning
  * Recent catalysts or events
- Analyze EPS trends and analyst sentiment
- Compare with historical price action

Use the news database to provide context on the broader market, vertical trends, and important insights from recent filings."""

messages = [
    {'role': 'system', 'content': system_message},
    {'role': 'user', 'content': 'What are the news, available EPS stats, and historical trend in the last 7 days for TSLA?'}
]
# messages = [{'role': 'user', 'content': 'What are the news, available EPS stats, and historical trend in the last 7 days for TSLA?'}]

In [37]:
while True:
    response: ChatResponse = chat(
        model='qwen3:32b',
        messages=messages,
        tools=[
            get_company_info, 
            get_eps_trend, 
            get_earnings_dates, 
            get_earnings_analysis, 
            get_historical_prices, 
            get_ticker_news,
            search_news_by_ticker,
            search_news_by_query,
            get_sec_filing
        ],
        think=True,
    )
    messages.append(response.message)
    print("Thinking: ", response.message.thinking)
    print("Content: ", response.message.content)
    if response.message.tool_calls:
        for tc in response.message.tool_calls:
            if tc.function.name in available_functions:
                print(f"Calling {tc.function.name} with arguments {tc.function.arguments}")
                result = available_functions[tc.function.name](**tc.function.arguments)
                print(f"Result: {result}")
                # add the tool result to the messages
                messages.append({'role': 'tool', 'tool_name': tc.function.name, 'content': str(result)})
    else:
        # end the loop when there are no more tool calls
        break
  # continue the loop with the updated messages


Thinking:  Okay, let's see. The user is asking about Tesla's recent news, available EPS stats, and the historical trend in the last 7 days. I need to break this down into parts.

First, for the news, the functions available are get_ticker_news and search_news_by_ticker. The get_ticker_news function gets recent articles from Yahoo Finance, while search_news_by_ticker might give more results from the indexed database. The user probably wants the most recent news, so maybe use get_ticker_news with a count of, say, 5. But the user might also want more comprehensive coverage, so including search_news_by_ticker with num_results=10 could help.

Next, EPS stats. The get_eps_trend function shows how analyst estimates have changed over time. The user is interested in the last 7 days, so this function should provide data points including 7daysAgo. Also, get_earnings_analysis might give a comprehensive view of EPS estimates and revisions. I should call both to get detailed info.

Historical trend 

## 2.1) Simple agent - no memory
* PAID Agent: based on OpenAI's technology for function calling: https://developers.openai.com/api/docs/guides/function-calling
* and using gpt-5.4-mini model (no thinking)
* you can use tool search (available from gpt-5.4), if you have many tools available (https://developers.openai.com/api/docs/guides/tools-tool-search)

In [38]:
# checking if openai package is installed and its version
!uv  pip list | grep openai

Using Python 3.12.7 environment at: /Users/realmistic/Documents/stocks-scoring-agent/.venv
openai                    2.13.0
openai-agents             0.6.3


In [39]:
import agents
print(f"📦 Version: {agents.__version__}")

📦 Version: 0.6.3


In [40]:
from agents import Agent, Runner

agent = Agent(name="Assistant", instructions="You are a helpful assistant")

result = await Runner.run(agent, "Write a haiku about recursion in programming.")
print(result.final_output)

Function calls itself,  
Solving problem step by step—  
Endless, yet precise.


In [41]:
from agents import function_tool

# in case we need WebSearch 
from agents import WebSearchTool

In [42]:
stock_agent = Agent(
    name='stock_agent',
    instructions="you're a helful assistant. Try to use all defined tools before going into the WebSearchTool",
    model='gpt-5.4-mini',
    tools=[ 
           WebSearchTool(), 
           function_tool(get_company_info),
           function_tool(get_eps_trend),
           function_tool(get_earnings_dates),
           function_tool(get_earnings_analysis),
           function_tool(get_historical_prices),
           function_tool(get_ticker_news),
           function_tool(search_news_by_ticker),
           function_tool(search_news_by_query),
           function_tool(search_companies),
           function_tool(get_top_value_companies),
           function_tool(get_top_growth_companies),
           function_tool(get_twitter_posts_by_engagement),
           function_tool(get_reddit_discussions_by_impact),
           function_tool(get_sec_filing)
           ] 
)

In [43]:
from agents import Runner

runner = Runner()

results = await runner.run(stock_agent, 
                           input="EPS trend for TSLA and the latest earnings call date and results, news sentiment")

In [44]:
print(results.final_output)

Here’s a quick TSLA snapshot based on the latest available data I found:

**EPS trend**
- **Current quarter (0q):** 0.45414
- **Next quarter (+1q):** 0.53750
- **Current year (0y):** 2.05937
- **Next year (+1y):** 2.49995

**Trend vs earlier estimates**
- Current quarter estimate is roughly flat over the last month, up slightly from 60 days ago, but down vs 90 days ago.
- Next quarter estimate has drifted slightly lower recently.
- Full-year estimates are basically stable to slightly down over the last 90 days.
- Next-year estimates have been revised down more noticeably over the last 60–90 days.

**Latest earnings call / result**
- **Latest reported earnings date:** **April 22, 2026**
- **EPS estimate:** 0.20
- **Reported EPS:** 0.13
- **Surprise:** **-36.54%**

**Next earnings date**
- **Expected earnings date:** **July 22, 2026**
- **EPS estimate:** 0.45

**Recent earnings history**
- Jan 28, 2026: 0.50 reported vs 0.45 estimate, **+10.96%**
- Oct 22, 2025: 0.50 reported vs 0.56 est

In [45]:
results.raw_responses

[ModelResponse(output=[ResponseFunctionToolCall(arguments='{"ticker":"TSLA"}', call_id='call_07iRnWS9FeKBPcIPX3Z6VFwP', name='get_eps_trend', type='function_call', id='fc_02a9e9af30851a77006a2f1d75947c81a394d1de64918a3b89', status='completed'), ResponseFunctionToolCall(arguments='{"ticker":"TSLA"}', call_id='call_n4FutLCt3zX88kaXpiEeCUjI', name='get_earnings_dates', type='function_call', id='fc_02a9e9af30851a77006a2f1d75949481a3847067dcb4fb4a25', status='completed'), ResponseFunctionToolCall(arguments='{"ticker":"TSLA","count":8}', call_id='call_qdlrNNUPgvQFVH5mFKco7E4Q', name='get_ticker_news', type='function_call', id='fc_02a9e9af30851a77006a2f1d7594a081a3977beb77e69c210a', status='completed')], usage=Usage(requests=1, input_tokens=6693, input_tokens_details=InputTokensDetails(cached_tokens=6528), output_tokens=76, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=6769, request_usage_entries=[]), response_id='resp_02a9e9af30851a77006a2f1d744c2c81a3952b6fdb6f

In [46]:
results.new_items

[ToolCallItem(agent=Agent(name='stock_agent', handoff_description=None, tools=[WebSearchTool(user_location=None, filters=None, search_context_size='medium'), FunctionTool(name='get_company_info', description='Get comprehensive company information and fundamental data for a stock ticker.\n\nReturns key metrics organized by category:\n- Company basics: website, industry, sector, employees, officers\n- Price data: current, previous close, day range, 52-week range\n- Market metrics: market cap, volume, beta, PE ratios\n- Valuation: margins, book value, price ratios\n- Ownership: insider/institutional holdings, short interest\n- Analyst data: EPS estimates, targets, recommendations\n- Financial health: cash, returns, growth rates', params_json_schema={'properties': {'ticker': {'description': "Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')", 'title': 'Ticker', 'type': 'string'}}, 'required': ['ticker'], 'title': 'get_company_info_args', 'type': 'object', 'additionalProperties': False}, 

In [47]:
pprint(results.new_items[0])

ToolCallItem(agent=Agent(name='stock_agent',
                         handoff_description=None,
                         tools=[WebSearchTool(user_location=None,
                                              filters=None,
                                              search_context_size='medium'),
                                FunctionTool(name='get_company_info',
                                             description='Get comprehensive '
                                                         'company information '
                                                         'and fundamental data '
                                                         'for a stock ticker.\n'
                                                         '\n'
                                                         'Returns key metrics '
                                                         'organized by '
                                                         'category:\n'
                         

In [48]:
# DEBUG: print everything for inspection (5k lines)
# from pprint import pprint
# pprint(vars(results))

In [49]:
# Write a wrapper functions to extract tool calls, outputs, and final message
                                                                                                                                                                
def get_tool_calls(results):                                                                                                                                       
    """Extract just the tool calls"""                                                                                                                              
    calls = []                                                                                                                                                     
    for response in results.raw_responses:                                                                                                                         
        for output in response.output:                                                                                                                             
            if hasattr(output, 'name') and hasattr(output, 'arguments'):                                                                                           
                calls.append({                                                                                                                                     
                    'name': output.name,                                                                                                                           
                    'arguments': output.arguments                                                                                                                  
                })                                                                                                                                                 
    return calls                                                                                                                                                   
                                                                                                                                                                    
def get_tool_outputs(results):                                                                                                                                     
    """Extract tool outputs with their names"""                                                                                                                    
    # Get tool calls from raw_responses                                                                                                                            
    calls = []                                                                                                                                                     
    for response in results.raw_responses:                                                                                                                         
        for output in response.output:                                                                                                                             
            if hasattr(output, 'name'):                                                                                                                            
                calls.append(output.name)                                                                                                                          
                                                                                                                                                                    
    # Get outputs from new_items                                                                                                                                   
    outputs = []                                                                                                                                                   
    for item in results.new_items:                                                                                                                                 
        if item.type == 'tool_call_output_item':                                                                                                                   
            outputs.append(item.output)                                                                                                                            
                                                                                                                                                                    
    return list(zip(calls, outputs))                                                                                                                               
                                                                                                                                                                    
def get_final_message(results):                                                                                                                                    
    """Get the final text response"""                                                                                                                              
    # Method 1: Use final_output                                                                                                                                   
    if results.final_output:                                                                                                                                       
        return results.final_output                                                                                                                                
                                                                                                                                                                    
    # Method 2: Extract from raw_responses                                                                                                                         
    for response in results.raw_responses:                                                                                                                         
        for output in response.output:                                                                                                                             
            if hasattr(output, 'content'):                                                                                                                         
                for content in output.content:                                                                                                                     
                    if hasattr(content, 'text'):                                                                                                                   
                        return content.text                        

In [50]:
# Usage:                                                                                                                                                           
calls = get_tool_calls(results)                                                                                                                                    
tool_outputs = get_tool_outputs(results)                                                                                                                           
final_msg = get_final_message(results)                                                                                                                             
                                                                                                                                                                    
print("=" * 70)                                                                                                                                                    
print("TOOL CALLS")                                                                                                                                                
print("=" * 70)                                                                                                                                                    
for i, call in enumerate(calls, 1):                                                                                                                                
    print(f"\n{i}. {call['name']}({call['arguments']})")                                                                                                           
                                                                                                                                                                    
print("\n" + "=" * 70)                                                                                                                                             
print("TOOL OUTPUTS")                                                                                                                                              
print("=" * 70)                                                                                                                                                    
for i, (name, output) in enumerate(tool_outputs, 1):                                                                                                               
    print(f"\n{i}. {name}")                                                                                                                                        
    print(f"   Output keys: {list(output.keys()) if isinstance(output, dict) else type(output)}")                                                                  
    if isinstance(output, dict) and 'ticker' in output:                                                                                                            
        print(f"   Ticker: {output['ticker']}")                                                                                                                    
        if 'data' in output:                                                                                                                                       
            data = output['data']                                                                                                                                  
            if isinstance(data, list):                                                                                                                             
                print(f"   Data items: {len(data)}")                                                                                                               
                                                                                                                                                                    
print("\n" + "=" * 70)                                                                                                                                             
print("FINAL MESSAGE")                                                                                                                                             
print("=" * 70)                                                                                                                                                    
print(f"\nLength: {len(final_msg) if final_msg else 0} characters")                                                                                                
if final_msg:                                                                                                                                                      
    # Print first 500 chars                                                                                                                                        
    print(f"\nPreview:\n{final_msg[:500]}...")              

TOOL CALLS

1. get_eps_trend({"ticker":"TSLA"})

2. get_earnings_dates({"ticker":"TSLA"})

3. get_ticker_news({"ticker":"TSLA","count":8})

TOOL OUTPUTS

1. get_eps_trend
   Output keys: ['ticker', 'data']
   Ticker: TSLA
   Data items: 4

2. get_earnings_dates
   Output keys: ['ticker', 'data']
   Ticker: TSLA
   Data items: 25

3. get_ticker_news
   Output keys: ['ticker', 'count', 'data']
   Ticker: TSLA
   Data items: 8

FINAL MESSAGE

Length: 1736 characters

Preview:
Here’s a quick TSLA snapshot based on the latest available data I found:

**EPS trend**
- **Current quarter (0q):** 0.45414
- **Next quarter (+1q):** 0.53750
- **Current year (0y):** 2.05937
- **Next year (+1y):** 2.49995

**Trend vs earlier estimates**
- Current quarter estimate is roughly flat over the last month, up slightly from 60 days ago, but down vs 90 days ago.
- Next quarter estimate has drifted slightly lower recently.
- Full-year estimates are basically stable to slightly down over t...


In [51]:
# Clean extraction                                                                                                                                                 
print("📋 TOOLS CALLED:")                                                                                                                                          
for resp in results.raw_responses:                                                                                                                                 
    for out in resp.output:                                                                                                                                        
        if hasattr(out, 'name'):                                                                                                                                   
            print(f"  • {out.name}({out.arguments})")                                                                                                              
                                                                                                                                                                    
print(f"\n📊 OUTPUTS: {len(results.new_items)} items")                                                                                                             
for item in results.new_items:                                                                                                                                     
    if item.type == 'tool_call_output_item':                                                                                                                       
        ticker = item.output.get('ticker', 'N/A')                                                                                                                  
        print(f"  • Ticker: {ticker}, Keys: {list(item.output.keys())}")                                                                                           
                                                                                                                                                                    
print(f"\n💬 FINAL OUTPUT: {len(results.final_output)} chars")                                                                                                     
print(results.final_output[:300] + "..." if len(results.final_output) > 300 else results.final_output)                                                             
           

📋 TOOLS CALLED:
  • get_eps_trend({"ticker":"TSLA"})
  • get_earnings_dates({"ticker":"TSLA"})
  • get_ticker_news({"ticker":"TSLA","count":8})

📊 OUTPUTS: 7 items
  • Ticker: TSLA, Keys: ['ticker', 'data']
  • Ticker: TSLA, Keys: ['ticker', 'data']
  • Ticker: TSLA, Keys: ['ticker', 'count', 'data']

💬 FINAL OUTPUT: 1736 chars
Here’s a quick TSLA snapshot based on the latest available data I found:

**EPS trend**
- **Current quarter (0q):** 0.45414
- **Next quarter (+1q):** 0.53750
- **Current year (0y):** 2.05937
- **Next year (+1y):** 2.49995

**Trend vs earlier estimates**
- Current quarter estimate is roughly flat ove...


In [52]:
from agents import RunResult
def get_usage_info_gpt_5_4_mini(result: RunResult):
    """Extract usage info from RunResult."""

    # https://developers.openai.com/api/docs/pricing
    # for 5.4-mini: $0.75 per 1M input tokens, $4.50 per 1M output tokens
    # for 5.4: $2.5 per 1M input tokens, $15 per 1M output tokens
    # for 5.5: $5 per 1M input tokens, $30 per 1M output tokens

    INPUT_PRICE = 0.75 / 1_000_000
    OUTPUT_PRICE = 4.50 / 1_000_000

    if hasattr(result, 'context_wrapper') and hasattr(result.context_wrapper, 'usage'):
        usage = result.context_wrapper.usage
        return {
            "requests": usage.requests,
            "input_tokens": usage.input_tokens,
            "output_tokens": usage.output_tokens,
            "total_tokens": usage.total_tokens,
            "request_usage_entries": [
                {
                    "input_tokens": entry.input_tokens,
                    "output_tokens": entry.output_tokens,
                    "total_tokens": entry.total_tokens 
                }
                for entry in usage.request_usage_entries
            ],
            "cost_usd": (
                usage.input_tokens * INPUT_PRICE +
                usage.output_tokens * OUTPUT_PRICE
            )
        }
    else:
        return {"error": "Usage information not available in the provided RunResult."}

In [53]:
# $0.016 USD
pprint(get_usage_info_gpt_5_4_mini(results))

{'cost_usd': 0.01469775,
 'input_tokens': 16135,
 'output_tokens': 577,
 'request_usage_entries': [{'input_tokens': 6693,
                            'output_tokens': 76,
                            'total_tokens': 6769},
                           {'input_tokens': 9442,
                            'output_tokens': 501,
                            'total_tokens': 9943}],
 'requests': 2,
 'total_tokens': 16712}


## 2.2) Agent with a Memory (passing context - RunResult)

In [54]:
# let's build a wrapper for calls AND PASS THE CONTEXT FOR NEXT QUESTIONS
                                                                                                                                                                
from typing import Optional                                                                                                                                        
from agents import Runner                                                                                                                                          
from agents.result import RunResult                                                                                                                                
from IPython.display import display, Markdown, HTML                                                                                                                
                                                                                                                                                                    
runner = Runner()                                                                                                                                                  
                                                                                                                                                                    
async def ask(agent, question: str, context: Optional[RunResult] = None) -> RunResult:                                                                             
    """Async ask with rich notebook output."""                                                                                                                     
                                                                                                                                                                    
    # Question header                                                                                                                                              
    display(HTML(f"<h3>❓ {question}</h3>"))                                                                                                                       
                                                                                                                                                                    
    # Run agent                                                                                                                                                    
    if context:                                                                                                                                                    
        results = await runner.run(agent, input=question, context=context.new_items)                                                                               
    else:                                                                                                                                                          
        results = await runner.run(agent, input=question)                                                                                                          
                                                                                                                                                                    
    # Show tools                                                                                                                                                   
    tools_html = "<div style='padding: 10px; border-radius: 5px;'>"                                                                     
    tools_html += "<b>🔧 Tools Called:</b><br>"                                                                                                                    
    tool_count = 0                                                                                                                                                 
    for resp in results.raw_responses:                                                                                                                             
        for out in resp.output:                                                                                                                                    
            if hasattr(out, 'name'):                                                                                                                               
                tool_count += 1                                                                                                                                    
                tools_html += f"&nbsp;&nbsp;{tool_count}. <code>{out.name}({out.arguments})</code><br>"                                                            
                                                                                                                                                                    
    if tool_count == 0:                                                                                                                                            
        tools_html += "&nbsp;&nbsp;<i>(No tools called)</i><br>"                                                                                                   
    tools_html += "</div>"                                                                                                                                         
                                                                                                                                                                    
    display(HTML(tools_html))                                                                                                                                      
                                                                                                                                                                    
    # Show response as markdown                                                                                                                                    
    display(Markdown("### 💬 Response:"))                                                                                                                          
    display(Markdown(results.final_output))                                                                                                                        
                                                                                                                                                                    
    return results                   

In [55]:
# Usage:
r1 = await ask(stock_agent, "What's TSLA's EPS trend and surprise historical, strategic developments, and recent social news?")                                                                                          

### 💬 Response:

Here’s a concise TSLA snapshot based on the latest analyst and social data I pulled.

## 1) EPS trend
**Quarterly EPS estimates are broadly stable to slightly down over the last 90 days.**

- **This quarter (0q):** current estimate **0.454** vs **0.462** 90 days ago
- **Next quarter (+1q):** current estimate **0.538** vs **0.541** 90 days ago
- **This year (0y):** current estimate **2.059** vs **2.089** 90 days ago
- **Next year (+1y):** current estimate **2.500** vs **2.810** 90 days ago

Interpretation: analysts are still projecting growth, but the forward trend has softened a bit, especially for next year. The consensus has been mostly flat over the past month, with a modest downward drift over 60–90 days.

## 2) Surprise history
Recent earnings surprises show **mixed performance**:

- **2025-06-30:** actual **0.40** vs est. **0.404** → **-1.0%**
- **2025-09-30:** actual **0.50** vs est. **0.559** → **-10.5%**
- **2025-12-31:** actual **0.50** vs est. **0.451** → **+11.0%**
- **2026-03-31:** actual **0.41** vs est. **0.350** → **+17.2%**

Takeaway: TSLA has had some volatility in meeting expectations, but the most recent two quarters came in above consensus.

## 3) Strategic / fundamental developments reflected in the data
From the news and social streams, the dominant strategic themes right now are:

- **SpaceX / Tesla merger speculation**
- **Elon Musk’s broader capital-structure and control narrative**
- **FSD approvals in Europe**
- **Robotaxi fleet growth in Texas**

These are mostly narrative-driven catalysts rather than hard financial disclosures, but they are clearly dominating investor attention right now.

## 4) Recent social news

### X / Twitter themes
The most engaged posts over the last week were overwhelmingly bullish and centered on:
- **Elon Musk becoming a trillionaire**
- **Tesla/SpaceX crossover speculation**
- **FSD approval in Denmark**
- **Texas Robotaxi fleet growth**
- **Retail trader success / “TSLA made me money” type posts**

### Reddit themes
High-impact Reddit discussion was dominated by:
- **SpaceX IPO**
- **Potential Tesla–SpaceX merger talk**
- **Valuation debates**
- Less focus on standalone TSLA fundamentals, more on Elon ecosystem speculation

## Bottom line
- **EPS trend:** stable but slightly weakening on the forward side
- **Earnings surprise history:** mixed, but recent quarters beat consensus
- **Strategic narrative:** FSD and robotaxi progress are the key operational bullish themes
- **Social sentiment:** very bullish, but heavily driven by Elon/SpaceX speculation rather than pure TSLA fundamentals

If you want, I can turn this into a **bull vs bear TSLA thesis** or a **one-page investor memo**.

In [56]:
# $0.018 per call
pprint(get_usage_info_gpt_5_4_mini(r1))

{'cost_usd': 0.0174165,
 'input_tokens': 18200,
 'output_tokens': 837,
 'request_usage_entries': [{'input_tokens': 6695,
                            'output_tokens': 149,
                            'total_tokens': 6844},
                           {'input_tokens': 11505,
                            'output_tokens': 688,
                            'total_tokens': 12193}],
 'requests': 2,
 'total_tokens': 19037}


In [57]:
r2 = await ask(stock_agent, "Find similar stocks to TSLA on vertical, market cap and P/E ratio", context=r1)                                                   

### 💬 Response:

Here are the **closest public-market peers to TSLA** I found using the dimensions you asked for: **vertical, market cap, and P/E ratio**.

### TSLA baseline
- **Tesla (TSLA)**
  - Market cap: about **$1.53T**
  - P/E: about **335.6**
  - Vertical: **EVs / autos / clean tech**
  - Operating margin: **5.9%**

### Best “similar” stocks by business model + scale
These are not perfect EV-only comps, but they’re the closest large-cap names with a mix of **tech/auto/platform-like scale** and high growth expectations:

| Ticker | Company | Why it’s somewhat similar | Market Cap | P/E |
|---|---|---|---:|---:|
| **NVDA** | NVIDIA | High-growth, innovation-led, premium valuation, mega-cap scale | ~$4.97T | ~41.6 |
| **AMD** | AMD | Growth/tech premium valuation, semiconductor platform company | ~$834B | ~194.3 |
| **GM** | General Motors | Auto vertical, but much cheaper valuation and lower growth profile | much smaller than TSLA | much lower than TSLA |
| **F** | Ford | Auto vertical, legacy manufacturer, not close on P/E | much smaller than TSLA | much lower than TSLA |
| **GE** | General Electric | Industrial transformation story, not auto/EV but similar “re-rating” theme | ~$350B | ~40.9 |
| **ARM** | Arm Holdings | High-growth, strategic platform company, very premium valuation | ~$407B | ~448.5 |
| **AAPL** | Apple | Hardware ecosystem, premium brand, huge market cap, but not auto | ~$4.28T | ~35.0 |
| **MSFT** | Microsoft | Mega-cap premium growth, though different vertical | ~$2.90T | ~23.0 |

### If you want the **closest match specifically on vertical**
For **automotive / EV / mobility**, the best public comps are:
- **GM**
- **F**
- **RIVN**
- **LCID**

But those are **not similar to TSLA on market cap or P/E**. TSLA is far larger and far more expensive on earnings than any of them.

### If you want the **closest match on market cap + P/E**
Then the better peers are more like:
- **NVDA**
- **ARM**
- **AMD**
- **GE**
- **AAPL**
- **MSFT**

But they are **not in Tesla’s vertical**.

### Bottom line
**Tesla is unusual** because it combines:
- **mega-cap scale**
- **EV/auto vertical**
- **extremely high P/E**

That combination makes it hard to find a true public-market peer.

If you want, I can do one of these next:
1. **Give you a ranked shortlist of 10 closest comps**
2. **Filter only EV/auto stocks**
3. **Filter only stocks with similar market cap and P/E**
4. **Build a similarity score for TSLA** using vertical + market cap + P/E

In [58]:
# $0.020 per call -- longer context
pprint(get_usage_info_gpt_5_4_mini(r2))

{'cost_usd': 0.02556,
 'input_tokens': 29082,
 'output_tokens': 833,
 'request_usage_entries': [{'input_tokens': 6692,
                            'output_tokens': 75,
                            'total_tokens': 6767},
                           {'input_tokens': 11100,
                            'output_tokens': 85,
                            'total_tokens': 11185},
                           {'input_tokens': 11290,
                            'output_tokens': 673,
                            'total_tokens': 11963}],
 'requests': 3,
 'total_tokens': 29915}


In [59]:
r3 = await ask(stock_agent, "What's Google's EPS trend vs. other top tech companies (magnificent 7)? For each list strategic insights from recent earnings calls and news.")                                                                                                              

### 💬 Response:

Here’s a compact EPS-trend comparison for **Google (GOOGL) vs. the rest of the Magnificent 7** using the latest analyst consensus data I pulled, plus a strategic read from recent earnings-history patterns and news flow.

## 1) EPS trend snapshot

**Current quarter EPS estimates (consensus):**
- **NVDA:** 2.08
- **GOOGL:** 2.87
- **MSFT:** 4.24
- **META:** 7.20
- **AMZN:** 1.82
- **AAPL:** 1.90
- **TSLA:** 0.45

**Next quarter EPS estimates:**
- **NVDA:** 2.35
- **GOOGL:** 3.00
- **MSFT:** 4.62
- **META:** 7.04
- **AMZN:** 1.91
- **AAPL:** 2.01
- **TSLA:** 0.54

**Full-year EPS estimates:**
- **NVDA:** 8.96
- **GOOGL:** 14.22
- **MSFT:** 16.84
- **META:** 32.83
- **AMZN:** 8.65
- **AAPL:** 8.76
- **TSLA:** 2.06

**Bottom line on Google:**  
GOOGL’s EPS trajectory is **solid but not the fastest** in the group. It sits in the middle-to-upper tier on absolute EPS, with **healthy year-over-year growth**: about **+24% this quarter** and **+31.6% for the full year**. That’s stronger than Apple, Amazon, and Microsoft on near-term quarterly growth, but far behind Nvidia and Meta on growth intensity. It’s also much less volatile than Tesla.  

## 2) How Google stacks up versus the Mag 7

### Google (GOOGL)
- **Consensus trend:** mostly stable to slightly improved over the last 90 days.
- **Revisions:** mixed short-term—some downward pressure in the last 7–30 days, but not dramatic.
- **Strategic read:** market still expects strong earnings power, but the stock is not getting the same “AI acceleration” estimate upgrades as Nvidia or Meta.

### Nvidia (NVDA)
- **Trend:** the clear standout.
- **Revisions:** very strong positive momentum across all periods.
- **Strategic read:** analysts are still aggressively lifting expectations, signaling the market sees sustained AI infrastructure demand.

### Meta (META)
- **Trend:** extremely high EPS base and strong annual growth.
- **Revisions:** mixed near-term, but the earnings base is massive.
- **Strategic read:** huge profitability, but next-quarter growth expectations are a lot more complicated and can swing with spending cycles.

### Microsoft (MSFT)
- **Trend:** steady, high-quality EPS growth.
- **Revisions:** recent near-term revisions look softer than some peers.
- **Strategic read:** durable cloud + AI monetization, but the estimate path is more “grind higher” than explosive.

### Amazon (AMZN)
- **Trend:** improving, especially on full-year EPS.
- **Revisions:** healthy for the current year, weaker for next quarter.
- **Strategic read:** investors are still balancing retail margin progress vs. heavy AI/cloud capex.

### Apple (AAPL)
- **Trend:** stable, modest growth.
- **Revisions:** surprisingly broad upward revision activity over 30 days.
- **Strategic read:** consensus is upbeat, but this is still the slow-growth member of the group.

### Tesla (TSLA)
- **Trend:** low absolute EPS, growth is present but not comparable to the software/AI names.
- **Revisions:** mixed and relatively soft.
- **Strategic read:** earnings remain highly sensitive to margin pressure, pricing, and product-cycle expectations.

## 3) Google’s strategic insights from recent earnings pattern + news

### What the earnings data suggests
- Google has delivered **big upside beats** recently, especially the most recent quarter in the dataset.
- Analysts are still expecting **~31.6% full-year EPS growth**, which is strong for a mega-cap.
- However, the **recent revisions are a little choppy**, implying the market may be pausing to assess how much AI investment and search/ads pressure will offset upside.

### What recent news suggests
Recent headlines around Google in the feed point to:
- **AI monetization and agentic shopping** as a major product/commerce theme.
- **Anthropic relying on Google infrastructure** for a large data-center push, reinforcing Google’s position as an AI infrastructure/platform player.
- Broader market narratives keep tying Google to the **AI bull market runway**.

### Strategic takeaway
Google looks like a **quality compounder with AI optionality**, not the most explosive EPS-growth story in Mag 7.  
The key question is whether:
1. **AI products meaningfully expand monetization**, and  
2. **capex/investment intensity stays manageable enough** to preserve margin expansion.

## 4) My ranking of the Mag 7 on EPS momentum right now

1. **NVDA** – strongest estimate momentum by far  
2. **META** – huge earnings power, still very strong growth  
3. **GOOGL** – solid growth + strong execution, but less revision momentum than the leaders  
4. **MSFT** – stable, premium-quality growth  
5. **AMZN** – improving, but next-quarter EPS is a softer spot  
6. **AAPL** – steady but slower growth profile  
7. **TSLA** – most volatile and least comparable on EPS scale

## 5) Short verdict on Google
If you’re comparing **EPS trend quality**, Google is:
- **better than Apple and Amazon on near-term growth**
- **much steadier than Tesla**
- **not as acceleration-heavy as Nvidia or Meta**
- **arguably the best “balanced” AI beneficiary** among the mega-cap platform names

If you want, I can turn this into a **clean comparison table with quarterly/yearly EPS growth, revision direction, and a one-line strategic thesis for each Magnificent 7 name**.

In [60]:
# $0.022 per call -- longer context
pprint(get_usage_info_gpt_5_4_mini(r3))

{'cost_usd': 0.02796,
 'input_tokens': 26948,
 'output_tokens': 1722,
 'request_usage_entries': [{'input_tokens': 6705,
                            'output_tokens': 412,
                            'total_tokens': 7117},
                           {'input_tokens': 20243,
                            'output_tokens': 1310,
                            'total_tokens': 21553}],
 'requests': 2,
 'total_tokens': 28670}


In [61]:
r4 = await ask(stock_agent, "Compare BIG 7 companies on valuations, analysts sentiment, investors demand, and growth prospects", context=r3)                                                                                             

### 💬 Response:

Here’s a concise comparison of the **Big 7** — **AAPL, MSFT, AMZN, GOOGL, META, NVDA, TSLA** — across **valuation, analyst sentiment, investor demand, and growth prospects**, using the latest data I pulled.

## Quick take
- **Cheapest on earnings:** **META** and **MSFT** look the most reasonable on trailing P/E.
- **Most expensive:** **TSLA** by a wide margin; **NVDA** is also rich, though backed by exceptional growth.
- **Strongest growth profile:** **NVDA** is the clear standout, with **GOOGL** and **MSFT** also strong.
- **Best investor demand / crowd interest:** **NVDA** and **TSLA** dominate recent Reddit discussion intensity; **MSFT** and **GOOGL** show strong fundamental interest.  
- **Most balanced overall:** **MSFT**.

## Snapshot table

| Ticker | Market cap | P/E (TTM) | Dividend yield | Operating margin | Analyst revisions / sentiment | Growth outlook |
|---|---:|---:|---:|---:|---|---|
| **AAPL** | $4.28T | 35.0 | 0.36% | 31.9% | Positive revisions, but mixed vs growth | Solid, but slower growth |
| **MSFT** | $2.90T | 23.0 | 0.68% | 48.5% | Mixed near-term, strong long-term | Strong and durable |
| **AMZN** | $2.57T | 27.9 | 0.0% | 13.6% | Generally favorable | Good, but near-term growth softer |
| **GOOGL** | n/a from search result | n/a | n/a | n/a | Strongly positive revisions and earnings surprises | Very strong |
| **META** | $1.44T | 20.3 | 0.37% | 42.2% | Bullish overall despite capex concerns | Strong |
| **NVDA** | $4.97T | 41.6 | 0.02% | 65.6% | Extremely bullish revisions | Outstanding |
| **TSLA** | $1.53T | 335.6 | 0.0% | 5.9% | Weakest of the group fundamentally | High upside, very high risk |

## 1) Valuations
**Lowest valuation risk:**
1. **META** — lowest P/E in the group from the data I pulled, with very high margins.
2. **MSFT** — still reasonable given profitability and consistency.
3. **AMZN** — higher than MSFT/META, but still not extreme.
4. **AAPL** — premium multiple for a mature company.
5. **NVDA** — expensive, but growth is extraordinary.
6. **TSLA** — farthest out on valuation by far.

**Interpretation:**  
- **META and MSFT** look most attractive if you care about paying a sensible price for quality.
- **NVDA** is expensive on traditional metrics, but its growth and margin profile are exceptional.
- **TSLA** is a story stock valuation, not a fundamentals-first valuation.

## 2) Analyst sentiment
Based on the earnings and revisions data:
- **Most bullish:** **NVDA**
  - Massive upward EPS revisions across all horizons.
  - Extremely strong earnings growth expectations.
- **Very positive:** **GOOGL**
  - Strong annual growth estimates and recent earnings beats.
- **Strong but mixed:** **MSFT**
  - Solid long-term estimates, though some near-term revisions have softened.
- **Good but slower:** **AAPL**
  - Positive revisions, but growth is more modest.
- **Still constructive:** **AMZN**
  - Strong annual growth, but next-quarter estimates are slightly negative.
- **Bullish, but with capex debate:** **META**
  - Analysts still see strong growth, though near-term EPS revisions are more mixed.
- **Weakest sentiment quality:** **TSLA**
  - Growth exists, but estimates and revisions are not in the same class as the others.

## 3) Investor demand
Using recent discussion intensity and engagement signals:
- **Highest retail/investor attention:** **NVDA** and **TSLA**
  - NVDA: huge engagement around AI demand and supply chain effects.
  - TSLA: intense debate, but much of it is speculative and valuation-driven.
- **Strong serious-investor interest:** **MSFT** and **GOOGL**
  - Discussion centered on valuation, durability, and AI monetization.
- **Solid but less explosive attention:** **AAPL**, **META**
  - AAPL conversation is split between AI progress and valuation concerns.
  - META interest is strong, but recent high-impact threads were harder to verify.
- **Less direct standalone crowd focus recently:** **AMZN**
  - Still discussed, but less as a standalone “hot” name than NVDA/TSLA.

## 4) Growth prospects
**Best growth prospects:**
1. **NVDA** — best by far on EPS growth.
2. **GOOGL** — very strong growth with improving margins and AI/Cloud optionality.
3. **MSFT** — durable growth from cloud, enterprise software, and AI.
4. **META** — strong earnings growth, though capex is a watch item.
5. **AMZN** — good growth, but near-term quarter looked softer.
6. **AAPL** — steady, but comparatively slower.
7. **TSLA** — still has long-term optionality, but fundamentals are weaker today.

## My ranking by category

### Best valuation
1. **META**
2. **MSFT**
3. **AMZN**
4. **AAPL**
5. **NVDA**
6. **TSLA**

### Best analyst sentiment
1. **NVDA**
2. **GOOGL**
3. **MSFT**
4. **META**
5. **AAPL**
6. **AMZN**
7. **TSLA**

### Best investor demand
1. **NVDA**
2. **TSLA**
3. **MSFT**
4. **GOOGL**
5. **AAPL**
6. **META**
7. **AMZN**

### Best growth prospects
1. **NVDA**
2. **GOOGL**
3. **MSFT**
4. **META**
5. **AMZN**
6. **AAPL**
7. **TSLA** on fundamentals alone, though it has the highest speculative upside

## Bottom line
- **Best all-around quality at a fair price:** **MSFT**
- **Best value among the mega-caps:** **META**
- **Best growth monster:** **NVDA**
- **Most controversial / speculative:** **TSLA**
- **Strongest long-term compounders besides NVDA:** **MSFT, GOOGL, META**

If you want, I can turn this into a **scorecard with numeric ratings out of 10** or a **“best pick by investor type”** version.

In [62]:
# $0.033 per call -- longer context
pprint(get_usage_info_gpt_5_4_mini(r4))

{'cost_usd': 0.03348375,
 'input_tokens': 30131,
 'output_tokens': 2419,
 'request_usage_entries': [{'input_tokens': 6694,
                            'output_tokens': 909,
                            'total_tokens': 7603},
                           {'input_tokens': 23437,
                            'output_tokens': 1510,
                            'total_tokens': 24947}],
 'requests': 2,
 'total_tokens': 32550}


In [63]:
r5 = await ask(stock_agent, "Find the most volatile stock from the previous context in the last 7 days with the largest price swings and news sentiment", context=r4)

### 💬 Response:

From the stocks in the previous context, **TSLA looks like the most volatile over the last 7 days**.

### Why TSLA
- **Largest daily swing range** among the names checked: on **June 5, 2026**, TSLA moved from a high of **424.68** to a low of **388.59** — a swing of about **8.8%** intraday.
- It also had another big range on **June 9, 2026**: **418.50** high vs **384.24** low — about **8.9%** intraday.
- By comparison, AAPL, MSFT, and NVDA all had smaller percentage swings over the same window.

### News sentiment
TSLA also had the **most intense news flow** and it was **mixed-to-bullish, but highly speculative**, which tends to amplify volatility:
- Multiple stories centered on **Tesla + SpaceX merger speculation**
- Coverage around **Elon Musk’s net worth / SpaceX IPO**
- Bullish analyst commentary on **self-driving progress**
- Some cautionary pieces about competition, robotics, and broader EV pressure

### Bottom line
**Most volatile stock: TSLA**
- **Price action:** widest swings in the 7-day period
- **Sentiment:** very active and mostly speculative, which likely contributed to the swings

If you want, I can also rank **all four stocks by volatility** with a quick table showing:
1. 7-day high-low range  
2. average daily swing  
3. sentiment tone (bullish/bearish/mixed)

In [64]:
# $0.04 cost for this call (with context)
pprint(get_usage_info_gpt_5_4_mini(r5))

{'cost_usd': 0.0293175,
 'input_tokens': 35676,
 'output_tokens': 569,
 'request_usage_entries': [{'input_tokens': 6700,
                            'output_tokens': 104,
                            'total_tokens': 6804},
                           {'input_tokens': 13097,
                            'output_tokens': 128,
                            'total_tokens': 13225},
                           {'input_tokens': 15879,
                            'output_tokens': 337,
                            'total_tokens': 16216}],
 'requests': 3,
 'total_tokens': 36245}


In [65]:
# Not working, as there is no VERTICAL stored in the database to compare with. The agent will fallback to WebSearchTool, which is not ideal for this type of query.
r6 = await ask(stock_agent, "Show top 10 growth companies in biotech sector. Try to find the companies in news if you can't get them otherwise")  

### 💬 Response:

Here are **10 biotech growth companies** I found from the latest biotech-related news coverage, since the broad company screen didn’t return biotech names cleanly:

1. **Eli Lilly (LLY)** — major growth story in obesity/weight-loss drugs; also expanding via acquisitions and new pipeline deals. citeturn0search0turn0search1  
2. **Vertex Pharmaceuticals (VRTX)** — established biotech with strong revenue, profitability, and pipeline expansion. citeturn0search1  
3. **Schrödinger (SDGR)** — AI-driven drug discovery platform plus real software revenue. citeturn0search0turn0search1  
4. **Sarepta Therapeutics (SRPT)** — commercial gene-therapy revenue with continued growth guidance. citeturn0search0  
5. **Viking Therapeutics (VKTX)** — obesity/weight-loss pipeline with bullish analyst upside. citeturn0search1  
6. **Relay Therapeutics (RLAY)** — promising trial data and a strong cash position. citeturn0search1  
7. **Nektar Therapeutics (NKTR)** — strong stock rally and upcoming Phase 3 catalyst. citeturn0search1  
8. **Abivax (ABVX)** — large stock move on strong ulcerative colitis trial results. citeturn0search1  
9. **CRISPR Therapeutics (CRSP)** — approved product plus deeper pipeline and cash strength. citeturn0search1  
10. **Vir Biotechnology (VIR)** — positive clinical catalysts and strong cash runway. citeturn0search1  

A quick note: this list is based on **recent biotech news and growth catalysts**, not a strict fundamentals-only screen of every biotech company. If you want, I can next turn this into a **ranked table with ticker, catalyst, revenue stage, and risk level**.

In [66]:
# same $0.029 per call (with no passed context)
pprint(get_usage_info_gpt_5_4_mini(r6))

{'cost_usd': 0.08080575,
 'input_tokens': 103253,
 'output_tokens': 748,
 'request_usage_entries': [{'input_tokens': 6700,
                            'output_tokens': 80,
                            'total_tokens': 6780},
                           {'input_tokens': 6808,
                            'output_tokens': 88,
                            'total_tokens': 6896},
                           {'input_tokens': 26026,
                            'output_tokens': 101,
                            'total_tokens': 26127},
                           {'input_tokens': 63719,
                            'output_tokens': 479,
                            'total_tokens': 64198}],
 'requests': 4,
 'total_tokens': 104001}


## 2.3) Agents with explicit context of a ticker (current_ticker) and a history

In [67]:
# automated context tracking and conversation history
class Conversation:                                                                                                                                                
    def __init__(self, agent, runner):                                                                                                                             
        self.agent = agent                                                                                                                                         
        self.runner = runner                                                                                                                                       
        self.history = []                                                                                                                                          
        self.current_ticker = None                                                                                                                                 
                                                                                                                                                                    
    async def ask(self, question: str) -> RunResult:                                                                                                               
        """Ask with automatic context and ticker tracking."""                                                                                                      
                                                                                                                                                                    
        # Extract ticker from question if present                                                                                                                  
        import re                                                                                                                                                  
        ticker_match = re.search(r'\b([A-Z]{1,5})\b', question)                                                                                                    
        if ticker_match:                                                                                                                                           
            self.current_ticker = ticker_match.group(1)                                                                                                            
                                                                                                                                                                    
        # Add ticker to question if we have context and it's not mentioned                                                                                         
        if self.current_ticker and self.current_ticker not in question.upper():                                                                                    
            question = f"{question} (for {self.current_ticker})"                                                                                                   
                                                                                                                                                                    
        print(f"\n{'='*70}")                                                                                                                                       
        print(f"❓ {question}")                                                                                                                                    
        if self.current_ticker:                                                                                                                                    
            print(f"🏷️  Ticker: {self.current_ticker}")                                                                                                           
        print('='*70)                                                                                                                                              
                                                                                                                                                                    
        # Run with context from previous question                                                                                                                  
        context = self.history[-1].new_items if self.history else None                                                                                             
                                                                                                                                                                    
        if context:                                                                                                                                                
            results = await self.runner.run(self.agent, input=question, context=context)                                                                           
        else:                                                                                                                                                      
            results = await self.runner.run(self.agent, input=question)                                                                                            

        # Show cost
        usage_info = get_usage_info_gpt_5_4_mini(results)
        print(f"\n💰 Cost: ${usage_info['cost_usd']:.6f}")

        # Show tools                                                                                                                                               
        print("\n🔧 Tools Called:")                                                                                                                                
        tool_count = 0                                                                                                                                             
        for resp in results.raw_responses:                                                                                                                         
            for out in resp.output:                                                                                                                                
                if hasattr(out, 'name'):                                                                                                                           
                    tool_count += 1                                                                                                                                
                    print(f"  {tool_count}. {out.name}({out.arguments})")                                                                                          
                                                                                                                                                                    
        if tool_count == 0:                                                                                                                                        
            print("  (No tools called)")                                                                                                                           
                                                                                                                                                                    
        # Extract ticker from results if not already set                                                                                                           
        if not self.current_ticker:                                                                                                                                
            for item in results.new_items:                                                                                                                         
                if item.type == 'tool_call_output_item' and isinstance(item.output, dict):                                                                         
                    if 'ticker' in item.output:                                                                                                                    
                        self.current_ticker = item.output['ticker']                                                                                                
                        break                                                                                                                                      
                                                                                                                                                                    
        # Show response                                                                                                                                            
        print(f"\n{'='*70}")                                                                                                                                       
        print("💬 Response:")                                                                                                                                      
        print('='*70)                                                                                                                                              
        print(results.final_output)                                                                                                                                
        print()                                                                                                                                                    
                                                                                                                                                                    
        self.history.append(results)                                                                                                                               
        return results                                                                                                                                             
                                                                                                                                                                    
    def reset(self):                                                                                                                                               
        """Clear conversation history."""                                                                                                                          
        self.history = []                                                                                                                                          
        self.current_ticker = None                                                                                                                                 
                                                                                                                                                                    


In [68]:
convo = Conversation(stock_agent, runner)   

await convo.ask("What's TSLA's EPS trend and surprise historical?")   


❓ What's TSLA's EPS trend and surprise historical?
🏷️  Ticker: TSLA

💰 Cost: $0.014072

🔧 Tools Called:
  1. get_eps_trend({"ticker":"TSLA"})
  2. get_earnings_analysis({"ticker":"TSLA"})
  3. get_earnings_dates({"ticker":"TSLA"})

💬 Response:
Here’s TSLA’s **EPS trend** and **historical earnings surprise** based on the latest available analyst data:

### EPS trend
- **Current quarter (0q):** **$0.454**
  - 7 days ago: $0.454
  - 30 days ago: $0.454
  - 60 days ago: $0.446
  - 90 days ago: $0.462
- **Next quarter (+1q):** **$0.538**
  - Slightly down from $0.539 7/30 days ago
- **Current year (0y):** **$2.059**
  - Up a bit from ~$2.049–$2.039 over the last 30–60 days
- **Next year (+1y):** **$2.500**
  - Down from ~$2.505 7 days ago and ~$2.772 60 days ago

### Historical EPS surprises
Recent quarterly surprises:
- **2026-04-22:** Estimate **$0.20**, reported **$0.13** → **-36.54%**
- **2026-01-28:** Estimate **$0.45**, reported **$0.50** → **+10.96%**
- **2025-10-22:** Estimate **$0

RunResult(input="What's TSLA's EPS trend and surprise historical?", new_items=[ToolCallItem(agent=Agent(name='stock_agent', handoff_description=None, tools=[WebSearchTool(user_location=None, filters=None, search_context_size='medium'), FunctionTool(name='get_company_info', description='Get comprehensive company information and fundamental data for a stock ticker.\n\nReturns key metrics organized by category:\n- Company basics: website, industry, sector, employees, officers\n- Price data: current, previous close, day range, 52-week range\n- Market metrics: market cap, volume, beta, PE ratios\n- Valuation: margins, book value, price ratios\n- Ownership: insider/institutional holdings, short interest\n- Analyst data: EPS estimates, targets, recommendations\n- Financial health: cash, returns, growth rates', params_json_schema={'properties': {'ticker': {'description': "Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')", 'title': 'Ticker', 'type': 'string'}}, 'required': ['ticker'], 'title

In [69]:
await convo.ask("What about analysts predictions of growth vs. index?")                                                                                            


❓ What about analysts predictions of growth vs. index? (for TSLA)
🏷️  Ticker: TSLA

💰 Cost: $0.012341

🔧 Tools Called:
  1. get_earnings_analysis({"ticker":"TSLA"})

💬 Response:
For **TSLA**, analysts currently expect **growth that is below the index in the near term, but above the index for the full year and next year**.

### TSLA growth vs. index
- **This quarter (0q):** TSLA **13.5%** vs index **25.98%**  
  - TSLA is **below** the index by about **12.5 percentage points**.
- **Next quarter (+1q):** TSLA **7.5%** vs index **20.28%**  
  - TSLA is **below** the index by about **12.8 points**.
- **This year (0y):** TSLA **24.06%** vs index **23.08%**  
  - TSLA is **slightly above** the index by about **1.0 point**.
- **Next year (+1y):** TSLA **21.39%** vs index **17.04%**  
  - TSLA is **above** the index by about **4.35 points**.

### Long-term growth
- **LTG:** TSLA’s long-term growth estimate isn’t available here, while the index is **12.2%**.

### Quick takeaway
Analysts are ba

RunResult(input='What about analysts predictions of growth vs. index? (for TSLA)', new_items=[ToolCallItem(agent=Agent(name='stock_agent', handoff_description=None, tools=[WebSearchTool(user_location=None, filters=None, search_context_size='medium'), FunctionTool(name='get_company_info', description='Get comprehensive company information and fundamental data for a stock ticker.\n\nReturns key metrics organized by category:\n- Company basics: website, industry, sector, employees, officers\n- Price data: current, previous close, day range, 52-week range\n- Market metrics: market cap, volume, beta, PE ratios\n- Valuation: margins, book value, price ratios\n- Ownership: insider/institutional holdings, short interest\n- Analyst data: EPS estimates, targets, recommendations\n- Financial health: cash, returns, growth rates', params_json_schema={'properties': {'ticker': {'description': "Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')", 'title': 'Ticker', 'type': 'string'}}, 'required': ['t

In [70]:
await convo.ask("Show extended analysis of competitors and strategic positioning?") 


❓ Show extended analysis of competitors and strategic positioning? (for TSLA)
🏷️  Ticker: TSLA

💰 Cost: $0.019628

🔧 Tools Called:
  1. get_company_info({"ticker":"TSLA"})
  2. get_top_growth_companies({"min_margin":0.15,"max_pe":null,"min_market_cap":1000000000,"limit":15})
  3. search_companies({"query":"Tesla competitors EV automakers","ticker":null,"min_market_cap":1000000000,"max_market_cap":null,"min_pe":null,"max_pe":null,"min_dividend":null,"max_dividend":null,"min_margin":null,"max_margin":null,"country":null,"limit":20})

💬 Response:
Here’s an extended competitor and strategic positioning view for **Tesla (TSLA)** based on the latest company/fundamental data I can access.

## 1) Tesla’s current positioning in one line
Tesla remains a **large-scale EV + energy platform company** with unusually high brand recognition and vertical integration, but its valuation still reflects a **growth/optionality premium** far beyond today’s current operating profitability. Its market cap is 

RunResult(input='Show extended analysis of competitors and strategic positioning? (for TSLA)', new_items=[ToolCallItem(agent=Agent(name='stock_agent', handoff_description=None, tools=[WebSearchTool(user_location=None, filters=None, search_context_size='medium'), FunctionTool(name='get_company_info', description='Get comprehensive company information and fundamental data for a stock ticker.\n\nReturns key metrics organized by category:\n- Company basics: website, industry, sector, employees, officers\n- Price data: current, previous close, day range, 52-week range\n- Market metrics: market cap, volume, beta, PE ratios\n- Valuation: margins, book value, price ratios\n- Ownership: insider/institutional holdings, short interest\n- Analyst data: EPS estimates, targets, recommendations\n- Financial health: cash, returns, growth rates', params_json_schema={'properties': {'ticker': {'description': "Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')", 'title': 'Ticker', 'type': 'string'}}, 're

In [71]:
await convo.ask("How is it doing against car manufacturers?") 


❓ How is it doing against car manufacturers? (for TSLA)
🏷️  Ticker: TSLA

💰 Cost: $0.013685

🔧 Tools Called:
  1. get_company_info({"ticker":"TSLA"})
  2. search_companies({"query":null,"ticker":null,"min_market_cap":null,"max_market_cap":null,"min_pe":null,"max_pe":null,"min_dividend":null,"max_dividend":null,"min_margin":null,"max_margin":null,"country":"USA","limit":20})

💬 Response:
Tesla is doing **better on profitability and scale than many traditional automakers**, but it’s also **much more expensive relative to those peers**.

### Quick take vs. car manufacturers
- **Scale:** Tesla’s market cap is about **$1.53T**, which is far above any pure-play automaker.
- **Margins:** Tesla’s **operating margin is ~4.2%** and **net margin ~4.0%**. That’s generally **stronger than many mass-market automakers**, though not extraordinary for a high-growth tech-style valuation.
- **Growth:** Revenue growth is about **15.8%**, which is typically **better than legacy automakers**.
- **Valuation

RunResult(input='How is it doing against car manufacturers? (for TSLA)', new_items=[ToolCallItem(agent=Agent(name='stock_agent', handoff_description=None, tools=[WebSearchTool(user_location=None, filters=None, search_context_size='medium'), FunctionTool(name='get_company_info', description='Get comprehensive company information and fundamental data for a stock ticker.\n\nReturns key metrics organized by category:\n- Company basics: website, industry, sector, employees, officers\n- Price data: current, previous close, day range, 52-week range\n- Market metrics: market cap, volume, beta, PE ratios\n- Valuation: margins, book value, price ratios\n- Ownership: insider/institutional holdings, short interest\n- Analyst data: EPS estimates, targets, recommendations\n- Financial health: cash, returns, growth rates', params_json_schema={'properties': {'ticker': {'description': "Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')", 'title': 'Ticker', 'type': 'string'}}, 'required': ['ticker'], '

In [72]:
await convo.ask("What about Google? Give me a comparative analysis similar to Tesla.") 


❓ What about Google? Give me a comparative analysis similar to Tesla. (for TSLA)
🏷️  Ticker: TSLA

💰 Cost: $0.044956

🔧 Tools Called:
  1. get_company_info({"ticker":"GOOGL"})
  2. get_earnings_analysis({"ticker":"GOOGL"})
  3. get_earnings_dates({"ticker":"GOOGL"})
  4. get_ticker_news({"ticker":"GOOGL","count":5})
  5. get_historical_prices({"ticker":"GOOGL","period":"1y","interval":"1d"})

💬 Response:
Here’s a **Tesla-style comparative read on Google / Alphabet (GOOGL)** based on the latest data I pulled.

## Quick verdict
**Alphabet looks like a high-quality large-cap growth compounder with much stronger profitability, balance sheet strength, and valuation support than Tesla.**  
If Tesla is a more speculative “future optionality” bet, Google is the more **cash-generative, diversified, lower-risk AI/platform monopoly** profile.

---

## Snapshot: Google vs. Tesla

| Metric | Google (GOOGL) | Tesla (TSLA) | Takeaway |
|---|---:|---:|---|
| Market cap | ~$4.39T | varies | Google is 

RunResult(input='What about Google? Give me a comparative analysis similar to Tesla. (for TSLA)', new_items=[ToolCallItem(agent=Agent(name='stock_agent', handoff_description=None, tools=[WebSearchTool(user_location=None, filters=None, search_context_size='medium'), FunctionTool(name='get_company_info', description='Get comprehensive company information and fundamental data for a stock ticker.\n\nReturns key metrics organized by category:\n- Company basics: website, industry, sector, employees, officers\n- Price data: current, previous close, day range, 52-week range\n- Market metrics: market cap, volume, beta, PE ratios\n- Valuation: margins, book value, price ratios\n- Ownership: insider/institutional holdings, short interest\n- Analyst data: EPS estimates, targets, recommendations\n- Financial health: cash, returns, growth rates', params_json_schema={'properties': {'ticker': {'description': "Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')", 'title': 'Ticker', 'type': 'string'}}, 

## 2.4) Stuctured output analysis

In [73]:
# STEP 1: Define simple Pydantic model with 5 main things
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum

class TrendDirection(str, Enum):
    IMPROVING = "improving"
    DECLINING = "declining" 
    STABLE = "stable"

class ValuationLevel(str, Enum):
    CHEAP = "cheap"
    FAIR = "fair"
    EXPENSIVE = "expensive"

class Recommendation(str, Enum):
    BUY = "buy"
    HOLD = "hold"
    SELL = "sell"

# 🎯 Simple model mapping 5 key metrics to our function tools
class SimpleStockAnalysis(BaseModel):
    """5 key metrics automatically mapped from function calls"""
    ticker: str = Field(description="Stock ticker symbol")
    company_name: str = Field(description="Company name")
    
    # 1. EPS Trend (from get_eps_trend, get_earnings_analysis)
    eps_current_estimate: Optional[float] = Field(description="Current quarter EPS estimate")
    eps_trend_direction: TrendDirection = Field(description="EPS trend direction")
    
    # 2. Price & Valuation (from get_company_info)
    current_price: Optional[float] = Field(description="Current stock price")
    pe_ratio: Optional[float] = Field(description="P/E ratio")
    distance_from_52w_high_pct: Optional[float] = Field(description="Distance from 52-week high %")
    valuation_level: ValuationLevel = Field(description="Valuation assessment")
    
    # 3. Analyst Sentiment (from get_earnings_analysis)
    analyst_revisions_net_7d: Optional[int] = Field(description="Net analyst revisions last 7 days")
    analyst_sentiment: TrendDirection = Field(description="Overall analyst sentiment")
    
    # 4. News & Market Activity (from get_ticker_news, get_historical_prices)
    recent_news_count: Optional[int] = Field(description="Number of recent news articles")
    news_sentiment_score: Optional[float] = Field(ge=-1, le=1, description="News sentiment (-1 to 1)")
    
    # 5. Investment Summary
    investment_thesis: str = Field(description="One-sentence investment summary")
    recommendation: Recommendation = Field(description="Buy/Hold/Sell recommendation")
    confidence_score: int = Field(ge=1, le=10, description="Confidence 1-10")

print("✅ Step 1 Complete: Pydantic model defined!")

✅ Step 1 Complete: Pydantic model defined!


In [74]:
# STEP 2: Build agent that outputs structured data
from agents import Agent, function_tool, WebSearchTool

structured_agent = Agent(
    name='structured_stock_agent',
    model='gpt-5.4-mini',  # Use gpt-5.4-mini for structured outputs
    instructions="""You are a stock analyst. Use the available tools to gather data, then provide a structured analysis.
    
    For each stock analysis:
    1. Get company info (price, PE ratio, 52-week high/low)
    2. Get EPS trends and analyst data 
    3. Get recent news
    4. Analyze the sentiment and trends
    5. Provide clear buy/hold/sell recommendation
    
    Be concise but thorough in your analysis.""",
    tools=[
        function_tool(get_company_info),
        function_tool(get_eps_trend), 
        function_tool(get_earnings_analysis),
        function_tool(get_ticker_news),
        WebSearchTool()
    ],
    output_type=SimpleStockAnalysis  # 🎯 This is the magic line!
)

print("✅ Step 2 Complete: Agent with structured output created!")


✅ Step 2 Complete: Agent with structured output created!


In [75]:
# STEP 3: Run analysis & get structured JSON data
from agents import Runner

async def get_structured_analysis(ticker: str) -> tuple[dict, float]:
    """Get structured analysis and cost."""
    runner = Runner()
    
    result = await runner.run(
        structured_agent, 
        f"Analyze {ticker} stock comprehensively using all available tools"
    )
    
    # Show tools called
    print("\n🔧 Tools Called:")
    tool_count = 0
    for resp in result.raw_responses:
        for out in resp.output:
            if hasattr(out, 'name'):
                tool_count += 1
                print(f"  {tool_count}. {out.name}({out.arguments})")
    
    if tool_count == 0:
        print("  (No tools called)")
    # Get cost
    cost = get_usage_info_gpt_5_4_mini(result)['cost_usd']
    
    # The structured output is automatically in result.final_output
    structured_data = result.final_output
    
    return structured_data, cost

# Test with TSLA
data, cost = await get_structured_analysis('TSLA')

print(f"✅ Step 3 Complete: Structured data retrieved!")
print(f"💰 Cost: ${cost:.6f}")
print(f"📊 Data type: {type(data)}")



🔧 Tools Called:
  1. get_company_info({"ticker":"TSLA"})
  2. get_eps_trend({"ticker":"TSLA"})
  3. get_earnings_analysis({"ticker":"TSLA"})
  4. get_ticker_news({"ticker":"TSLA","count":5})
✅ Step 3 Complete: Structured data retrieved!
💰 Cost: $0.012643
📊 Data type: <class '__main__.SimpleStockAnalysis'>


In [76]:
# STEP 4.1: Print structured data as clean JSON - Option 1: Use Pydantic's built-in JSON serialization (BEST)
import json

print("✅ Step 4: Clean JSON Output")
print("=" * 50)

# Option 1: Use Pydantic's built-in JSON serialization (BEST)
print(data.model_dump_json(indent=2))

# Option 2: Convert to dict first, then use json.dumps
# print(json.dumps(data.model_dump(), indent=2))

print("=" * 50)


✅ Step 4: Clean JSON Output
{
  "ticker": "TSLA",
  "company_name": "Tesla, Inc.",
  "eps_current_estimate": 0.45414,
  "eps_trend_direction": "stable",
  "current_price": 406.43,
  "pe_ratio": 369.4818,
  "distance_from_52w_high_pct": 18.52,
  "valuation_level": "expensive",
  "analyst_revisions_net_7d": 1,
  "analyst_sentiment": "stable",
  "recent_news_count": 5,
  "news_sentiment_score": 0.2,
  "investment_thesis": "Tesla has strong revenue and EPS growth, ample liquidity, and a favorable long-term strategic position, but its valuation is extremely rich and recent estimate revisions are mixed, making the stock attractive only for investors comfortable with high execution risk.",
  "recommendation": "hold",
  "confidence_score": 7
}


In [77]:
# STEP 4.2: Print structured data as clean JSON - Option 2: Convert to dict

from pprint import pprint

# Convert to dict and pretty print
pprint(data.model_dump())

{'analyst_revisions_net_7d': 1,
 'analyst_sentiment': <TrendDirection.STABLE: 'stable'>,
 'company_name': 'Tesla, Inc.',
 'confidence_score': 7,
 'current_price': 406.43,
 'distance_from_52w_high_pct': 18.52,
 'eps_current_estimate': 0.45414,
 'eps_trend_direction': <TrendDirection.STABLE: 'stable'>,
 'investment_thesis': 'Tesla has strong revenue and EPS growth, ample '
                      'liquidity, and a favorable long-term strategic '
                      'position, but its valuation is extremely rich and '
                      'recent estimate revisions are mixed, making the stock '
                      'attractive only for investors comfortable with high '
                      'execution risk.',
 'news_sentiment_score': 0.2,
 'pe_ratio': 369.4818,
 'recent_news_count': 5,
 'recommendation': <Recommendation.HOLD: 'hold'>,
 'ticker': 'TSLA',
 'valuation_level': <ValuationLevel.EXPENSIVE: 'expensive'>}


In [78]:
# The output is already structured as a Pydantic model, so we can directly pretty print it

# compare vs class definition: 
# 🎯 Simple model mapping 5 key metrics to our function tools
# class SimpleStockAnalysis(BaseModel):
#     """5 key metrics automatically mapped from function calls"""
#     ticker: str = Field(description="Stock ticker symbol")
#     company_name: str = Field(description="Company name")
    
#     # 1. EPS Trend (from get_eps_trend, get_earnings_analysis)
#     eps_current_estimate: Optional[float] = Field(description="Current quarter EPS estimate")
#     eps_trend_direction: TrendDirection = Field(description="EPS trend direction")
    
#     # 2. Price & Valuation (from get_company_info)
#     current_price: Optional[float] = Field(description="Current stock price")
#     pe_ratio: Optional[float] = Field(description="P/E ratio")
#     distance_from_52w_high_pct: Optional[float] = Field(description="Distance from 52-week high %")
#     valuation_level: ValuationLevel = Field(description="Valuation assessment")
    
#     # 3. Analyst Sentiment (from get_earnings_analysis)
#     analyst_revisions_net_7d: Optional[int] = Field(description="Net analyst revisions last 7 days")
#     analyst_sentiment: TrendDirection = Field(description="Overall analyst sentiment")
    
#     # 4. News & Market Activity (from get_ticker_news, get_historical_prices)
#     recent_news_count: Optional[int] = Field(description="Number of recent news articles")
#     news_sentiment_score: Optional[float] = Field(ge=-1, le=1, description="News sentiment (-1 to 1)")
    
#     # 5. Investment Summary
#     investment_thesis: str = Field(description="One-sentence investment summary")
#     recommendation: Recommendation = Field(description="Buy/Hold/Sell recommendation")
#     confidence_score: int = Field(ge=1, le=10, description="Confidence 1-10")
pprint(data)

SimpleStockAnalysis(ticker='TSLA', company_name='Tesla, Inc.', eps_current_estimate=0.45414, eps_trend_direction=<TrendDirection.STABLE: 'stable'>, current_price=406.43, pe_ratio=369.4818, distance_from_52w_high_pct=18.52, valuation_level=<ValuationLevel.EXPENSIVE: 'expensive'>, analyst_revisions_net_7d=1, analyst_sentiment=<TrendDirection.STABLE: 'stable'>, recent_news_count=5, news_sentiment_score=0.2, investment_thesis='Tesla has strong revenue and EPS growth, ample liquidity, and a favorable long-term strategic position, but its valuation is extremely rich and recent estimate revisions are mixed, making the stock attractive only for investors comfortable with high execution risk.', recommendation=<Recommendation.HOLD: 'hold'>, confidence_score=7)
